# GeoNUSAF Part 2 / Stage 1 — Unconditional mask DDPM (one-hot labels, v-prediction)

**Kathmandu Valley land-use layout generation — 6 classes + ignore, 128x128**

Structural twin of `segformer-b0-new.ipynb`: same 25 sections, same config-dataclass pattern, same
asserts, same persistence. Only the modelling sections (9, 11, 12, 13, 17-19) differ, so the two
notebooks diff cleanly. Everything about the dataset, the filename convention, the colour LUT and
the fold split is carried over unchanged from the segmentation notebooks — deliberately, because
Stage 2 has to land on exactly the same tiles.

## This is arm B of a two-arm ablation
Run A (`geonusaf-stage1-maskddpm-block-fold1`) used the SatSynth **binary label code** with
**epsilon** prediction. It trained cleanly — validation eps-MSE 0.00380 at epoch 536, gradient norms
stable, no visible overfitting — and produced unusable masks. Measured on its epoch-699 grid against
the fold-1 train marginal:

| class | real % | run A % | ratio |
|---|---|---|---|
| Residential | 52.61 | 42.99 | 0.82 |
| Road | 1.43 | 1.22 | 0.85 |
| **River** | **0.47** | **3.62** | **7.70** |
| **Forest** | **12.41** | **6.76** | **0.54** |
| UnusedLand | 14.27 | 15.04 | 1.05 |
| Agricultural | 18.53 | 29.99 | 1.62 |
| ignore | 0.27 | 0.38 | 1.41 |

illegal-code rate 0.479%. Rivers came out as ponds rather than channels.

**Two causes, both fixed here, neither of them model capacity.**

1. **The binary code hallucinates classes at boundaries.** River (010) is the exact midpoint of
   Residential (000) and Forest (011) — the most common boundary in the dataset. The three channels
   are thresholded independently, so wherever one bit crosses zero before the other, the decoder
   writes River. Agricultural (101) is the midpoint of Road (001) and UnusedLand (100); Forest (011)
   meeting Agricultural (101) has the illegal code 111 as its midpoint. Every over-produced class
   in the table above is a midpoint; every under-produced one is a boundary donor. Section 9 has
   the arithmetic and a side-by-side degradation test.
2. **Zero terminal SNR with epsilon prediction is degenerate.** At `alpha_bar_T = 0` the epsilon
   target equals the network input, so the model cannot learn the global mean and the marginals
   come out flattened toward uniform — which is the Residential 0.82x / rare-class inflation
   pattern, and it is also what run A's own header describes happening before the zero-SNR rescale
   was added. Lin et al. (WACV 2024) prescribe **v-prediction** for this case. Section 11 asserts
   the broken pair can never be trained again.

Keep run A. Two arms, one representation change, a mechanistic explanation and a measurable
outcome is a paper result, not a bug to hide.

## Why fold 1
Measured class shares over the whole dataset: Residential 56.03, Agricultural 18.10, UnusedLand
12.96, Forest 10.39, Road 1.57, River 0.60, ignore 0.35. **The 42.31 / 20.74 / 19.56 figures on the
old spec sheet are wrong** and should be corrected wherever they appear.

| fold | val River | worst train->val shift | verdict |
|---|---|---|---|
| 0 | **0.04%** (~105 px per 512 mask) | Forest 5.4x | unusable — nothing to score River against |
| **1** | **0.86%** | Forest 0.53x | **mildest shift on every class; best mIoU in part 1** |
| 2 | 0.88% | Forest **0.15x** | val block is 2.24% Forest against 14.54% in train |

Fold 1 train: 52.61 / 1.43 / 0.47 / 12.41 / 14.27 / 18.53 / 0.27. Fold 1 val: 64.17 / 1.88 / 0.86 /
6.52 / 9.80 / 16.27 / 0.50. The generator sees the train half only, and the stage-4 augmentation
experiment must run on **this same fold**.

## What this stage produces
An unconditional diffusion model over **masks only**. It samples new, plausible land-use layouts
that never existed in the dataset. Each sampled mask is written as a single-channel PNG at 512x512
and handed to Stage 2, which paints a satellite image onto it. The pair (fake image, fake mask) is
then mixed into part-1 segmentation training — the point of the whole exercise is the rare classes,
**River (0.47% of fold-1 train pixels), Road (1.43%) and UnusedLand (IoU ~0.18-0.30 in part 1)**.

## What is fixed by protocol (do not change per-model)
| Item | Value |
|---|---|
| Dataset | 408 base tiles x 3 flip variants = 1224 pairs, unchangeable |
| Split | fold **1** of the 3-fold sequence-block CV, `BLOCK_BUFFER=2` dropped from train only |
| Leakage rule | the generator trains on **fold-1 train tiles only**; val tiles never enter a gradient |
| Colour -> index | LUT applied at **full 2345x2351 resolution**, then `INTER_NEAREST` |
| Ignore | index 255 on disk, carried as code 6 inside the model |
| Augmentation | rot90 (k in 0..3) only; hflip/vflip are baked into the 1224 files and are NOT repeated |
| Seed | single seed = 42 |
| Determinism | **off** (see section 3) |
| Runtime | Kaggle T4 x1, AMP fp16, 12 h session cap. **~20 h for 2400 epochs = 2-3 sessions.** |
| Effective batch | 32 (`BATCH_SIZE` x `GRAD_ACCUM`; the split is a memory knob, the product is not) |
| Output contract | `genmask{i}.png`, single channel, values `{0..5, 255}`, 512x512, `INTER_NEAREST` |

## What is model-specific here
- **One-hot label channels.** 7 codes, 7 channels, `+1` on the true class and `-1` elsewhere,
  `argmax` to decode. Four extra channels against the binary code, under 0.1% of the parameters,
  and no illegal state to project away. See section 9.
- **v-prediction**, T=1000, linear beta 1e-4 -> 2e-2, zero terminal SNR, **offset noise off**
  (on one-hot channels the per-channel mean *is* the class marginal, so training with an offset and
  sampling without one biases class balance directly).
- **`diffusers.UNet2DModel`, ~12 M params** (`BASE_CH=64`), attention at 32 and 16. Run A's
  "45 M overfit at epoch 550" finding came from the same val eps-MSE that also selected an unusable
  checkpoint, so treat it as unproven. Capacity is not the bottleneck; keep 12 M for this arm so
  the two arms differ in one thing.
- **60 k steps, not 15 k.** Run A did 25 steps/epoch x 600 epochs = 15 k, which is very short for a
  diffusion model trained from scratch.
- **Model selection on sampled class-share KL**, never on the denoising loss. See section 17.
- **128x128 only.** At 128 a River is ~3 px wide and a Road ~2 px, which is tight — but fix the
  representation before spending 4x the compute at 256, or you will render the same artefact in
  more detail.

## How this run is judged
There is no mIoU here. Five numbers decide whether the samples are usable, all in sections 19 and 24:
1. class-proportion **KL** against the real fold-1 train masks, reported filtered *and* unfiltered,
2. **share ratio for River and Road** — bounded on both sides now, because run A passed the old
   one-sided test while over-producing River 7.7x,
3. **connectivity of Road and River** — components per 1000 class pixels,
4. landscape metrics — patch density, edge density, mean patch area, Shannon diversity,
5. a nearest-neighbour memorization check, plus the reconstruction check that says whether a bad
   grid means "undertrained" or "wrong prior".

## Session strategy (12 h Kaggle cap)
Full optimizer/scaler/scheduler/EMA state is pushed to Hugging Face as `last.pt`; on restart the
notebook resumes from it automatically. `MAX_HOURS` stops the loop before Kaggle kills the session,
so a checkpoint always survives. Expect to run this notebook 2-3 times.

## Prerequisites
- **Internet ON** (Settings -> Internet). Needed for pip, W&B, HF.
- Kaggle Secrets: `HF_TOKEN` (write scope) and `WANDB_API_KEY`.
- GPU: T4 x1.


## 1. Environment

`diffusers` is not on the Kaggle image; `ptflops` is not either. Versions are printed rather than
pinned beyond a floor — `UNet2DModel` and `DDIMScheduler` have had a stable API since 0.15, but
record the printed version in the paper's reproducibility section.

In [ ]:
!pip install -q "diffusers>=0.27.0" ptflops==0.7.3 2>&1 | tail -2

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # set BEFORE cuda initialises

import re, gc, json, time, math, random, shutil, warnings, itertools, copy, hashlib, tarfile
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

import diffusers
print("torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

**Stage 1 runs on fold 0 only.** That is not a convenience, it is the leakage rule: the generator
may only ever see fold-0 training tiles, because the augmentation experiment in stage 4 is reported
on fold 0. Training on all 408 tiles and then evaluating on a held-out fold would make every
downstream number meaningless, so `FOLD` and `SPLIT_MODE` are here for parity with the segmentation
notebooks rather than as knobs to sweep.

The label-coding block and the diffusion block are grouped separately so that an ablation
("one-hot instead of binary", "x0 instead of epsilon") is a one-line change rather than a code edit.

In [ ]:
@dataclass
class CFG:
    # ---- run selection (fixed for stage 1 — see the leakage rule) ----
    SPLIT_MODE: str = "block"          # stage 1 is block-only; "random" would break the protocol
    FOLD: int = 1                      # fold 1: the only fold whose val block can measure River

    # ---- label representation (THE change vs the binary-code run) ----
    # "onehot": 7 channels, +1 on the true class, -1 elsewhere, argmax to decode.
    # The binary-code run is kept as ablation A; this is ablation B. See section 9.
    LABEL_CODING: str = "onehot"

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/satellite-datasets/satellite_dataset"   # auto-detected below
    CACHE_DIR: str = "/kaggle/working/cache_idx"
    # If you saved the index cache as a Kaggle dataset, point this at it to skip the repack.
    PREBUILT_CACHE: Optional[str] = None
    GEN_SIZE: int = 128                # diffusion resolution
    OUT_SIZE: int = 512                # interface resolution handed to Stage 2
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255            # on disk, and in the Stage-2 contract
    IGNORE_CODE: int = 6               # inside the model
    N_CODES: int = 7                   # 0..5 + ignore
    N_CH: int = 7                      # one channel per code. Was 3 (ceil(log2 7)) in run A.
    PRELOAD_RAM: bool = True           # ~20 MB for the whole 128 cache

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary

    # ---- diffusion ----
    TIMESTEPS: int = 1000
    BETA_START: float = 1e-4
    BETA_END: float = 2e-2
    # v-prediction, NOT epsilon. Zero terminal SNR sets alpha_bar_T = 0, at which point the
    # epsilon target is identical to the network input and carries no information -- the model
    # cannot learn the global mean, and the class marginals come out flattened toward uniform
    # (run A: trained on 52.6% Residential, generated 43.0%; River 0.47 -> 3.62%). Lin et al.
    # (WACV 2024) pair zero terminal SNR with v-prediction for exactly this reason. The assert in
    # section 11 refuses the broken combination.
    PREDICTION_TYPE: str = "v_prediction"
    DDIM_STEPS: int = 50
    DDIM_ETA: float = 0.0
    ZERO_TERMINAL_SNR: bool = True     # rescale betas so alpha_bar_T == 0 (Lin et al., WACV 2024)
    # Offset noise is OFF. In an RGB model it widens the brightness range; here the per-channel
    # mean IS the class marginal, and training with an offset while sampling from plain randn is a
    # train/sample mismatch that biases class balance directly. Run A had this at 0.1.
    OFFSET_NOISE: float = 0.0

    # ---- model (UNet2DModel) ----
    BASE_CH: int = 64                  # ~12 M params
    CH_MULT: tuple = (1, 2, 2, 4)      # -> (64, 128, 128, 256) at 128/64/32/16
    LAYERS_PER_BLOCK: int = 2
    ATTN_RES: tuple = (32, 16)         # attention at these spatial resolutions
    NORM_GROUPS: int = 32
    ATTN_HEAD_DIM: int = 8

    # ---- optimization ----
    # Effective batch is BATCH_SIZE x GRAD_ACCUM = 32 and is part of the protocol; the split
    # between the two is a pure memory knob. See section 12.
    BATCH_SIZE: int = 32               # 12 M params fits batch 32 without checkpointing
    GRAD_ACCUM: int = 1                # -> effective batch 32
    LR: float = 2e-4                   # DDPM default for a UNet of this size
    WEIGHT_DECAY: float = 0.0          # diffusion UNets are trained without decay
    NO_DECAY_ON_NORM_BIAS: bool = True
    # 25 steps/epoch. Run A did 600 epochs = 15 k steps, which is very short for a diffusion model
    # trained from scratch. 2400 epochs = 60 k steps, about 20 h, so 2-3 Kaggle sessions with the
    # automatic resume in section 18. If you are time-boxed, 1200 (30 k steps) fits one session
    # and is the minimum I would report -- set COS_HORIZON_EPOCHS to match, the cosine must finish.
    MAX_EPOCHS: int = 1600
    WARMUP_STEPS: int = 500
    CLIP_NORM: float = 1.0
    COS_HORIZON_EPOCHS: int = 1600     # must equal MAX_EPOCHS: the run should finish the cosine
    USE_EMA: bool = True               # sampling always uses the EMA weights
    EMA_DECAY: float = 0.9995

    # ---- augmentation (masks only — no photometric term exists) ----
    AUG_ROT90: bool = True
    # Oversampling of tiles that contain River, as a weight multiplier. DEFAULT OFF, deliberately:
    # run A over-produced River by 7.7x, so oversampling would have made it worse. Pull this lever
    # only if section 18's sel_share_River comes out BELOW the real 0.47% once the representation
    # is fixed. It biases the learned marginal, so any run using it must say so in the paper.
    RIVER_OVERSAMPLE: float = 0.0

    # ---- monitoring, model selection and generation ----
    VAL_MONITOR: bool = True           # v-MSE on val tiles, NO gradient — a curve, not the selector
    VAL_NOISE_SEED: int = 1234         # fixed noise so the val curve is comparable across epochs
    VAL_LOWT: int = 200                # also report v-MSE restricted to t < this (detail regime)
    # Model selection is on SAMPLE quality, not on the denoising loss. Run A selected e536 on
    # val eps-MSE 0.00380 and produced unusable samples: the loss at random t is dominated by the
    # high-t regime and is nearly blind to what the sampler actually emits.
    SELECT_ON: str = "kl"              # "kl" (class-share KL vs real) or "val_loss" (run-A behaviour)
    SEL_N_SAMPLE: int = 128            # masks drawn at each selection point (was 64: too noisy --
                                       # KL's own sd matched the best value, so 'best' was a draw)
    SEL_DDIM_STEPS: int = 50           # cheap steps for selection; the hand-off set uses more
    SEL_WARMUP_EPOCHS: int = 200       # do not select before this; early KL is noise
    SAMPLE_EVERY: int = 50             # epochs between selection points (doubled to pay for the
                                       # doubled SEL_N_SAMPLE; net sampling cost unchanged)
    N_SAMPLE_GRID: int = 16
    RECON_CHECK: bool = True           # noise real masks to t, denoise back, report pixel accuracy
    # Admissibility gate on selection. KL is dominated by the big classes -- Road and River together
    # are only ~8% of it -- so a checkpoint that has quietly stopped drawing rivers can still post
    # the best KL. This does NOT change the metric (KL stays comparable across arms); it only
    # refuses to crown a checkpoint whose River share is outside this band around the real value.
    RIVER_GATE_LO: float = 0.40
    RIVER_GATE_HI: float = 2.00
    # One-shot, for the resume where SEL_N_SAMPLE changes. A 64-sample KL and a 128-sample KL are
    # not the same estimator: the old best (0.00960 @ e999) was a lucky low draw that a lower-
    # variance estimator may never beat, which would freeze best.pt for the rest of the run.
    RESET_BEST_ON_RESUME: bool = True
    RECON_T: int = 500
    N_POOL: int = 3500                 # masks sampled, before filtering. 1500 -> 1000 gave the
                                       # distribution-matched filter almost no selection pressure;
                                       # 3500 -> 1000 lets it keep the River-bearing masks
    N_GENERATE: int = 1000             # size of the Stage-2 hand-off set, selected from the pool
    MIN_COMPONENT_PX: int = 12         # at 128: drop Road/River blobs smaller than this
    # Post-processing. The old despeckle only DELETED small Road/River blobs; it could not
    # reconnect a river broken by a 1 px gap, which is what River's frag_ratio 3.59 was made of.
    # smooth_mask adds a modal filter on the bulk classes and a binary closing on the thin ones.
    # The source masks are hand-digitized polygons, piecewise smooth by construction; the model
    # emits pixel-level boundary noise the real data never had. Declare this in the paper.
    PP_SWEEP: bool = True              # sweep the four knobs on a subsample and auto-pick
    PP_SWEEP_N: int = 600
    PP_MODAL: int = 3                  # fallback values if the sweep is off or finds nothing
    PP_CLOSE: int = 3
    PP_THIN: int = 30
    PP_BULK: int = 0
    FILTER_MASKS: bool = True          # despeckle + distribution-matched selection
    GEN_BATCH: int = 25                # sampling sets the VRAM peak, not training -- keep it modest
    FINAL_DDIM_STEPS: int = 200        # more steps for the hand-off set than for the training grids
    EVAL_N_REAL: int = 900             # >= 804 means ALL fold-train tiles. A 400-tile
                                       # subset had River at 0.376% vs the true 0.47%,
                                       # a 20% error on the class under scrutiny.
    NN_CHECK_N: int = 200              # fakes scored in the memorization check
    AMB_MARGIN: float = 0.25           # top1 - top2 below this counts as an ambiguous pixel

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 2
    AMP: bool = True
    MAX_HOURS: float = 10.5            # stop before Kaggle's 12 h cap
    GRAD_CHECKPOINT: bool = False      # not needed at 12 M params; ~30% faster without it
    MULTI_GPU: bool = False            # opt-in: DataParallel over "T4 x2". See section 12.

    # ---- persistence ----
    HF_USER: str = "sugam24"
    HF_PREFIX: str = "geonusaf-stage1-maskddpm-onehot"   # separate repo from the binary-code run
    PUSH_LAST_EVERY: int = 25          # epochs between full-state pushes (~500 MB each, so not 10)
    WANDB_PROJECT: str = "geonusaf-baselines"
    OUT_DIR: str = "/kaggle/working/out"
    GEN_DIR: str = "/kaggle/working/gen_masks_512"

    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.GEN_DIR, exist_ok=True)

assert cfg.LABEL_CODING == "onehot", "this notebook is the one-hot arm; the binary arm is run A"
assert cfg.N_CH == cfg.N_CODES, "one-hot needs exactly one channel per code"
assert cfg.COS_HORIZON_EPOCHS == cfg.MAX_EPOCHS, "the cosine must finish inside the run"

CODE_NAMES = list(cfg.CLASS_NAMES) + ["Ignore"]         # 7 code slots, index == code
RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
tag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print(f"diffusion: T={cfg.TIMESTEPS} beta=[{cfg.BETA_START},{cfg.BETA_END}] "
      f"pred={cfg.PREDICTION_TYPE} zsnr={cfg.ZERO_TERMINAL_SNR} offset={cfg.OFFSET_NOISE} "
      f"ddim={cfg.DDIM_STEPS} | labels: {cfg.LABEL_CODING}, {cfg.N_CH} channels / "
      f"{cfg.N_CODES} codes @ {cfg.GEN_SIZE}px")
print(f"batch: {cfg.BATCH_SIZE} x {cfg.GRAD_ACCUM} accum = {cfg.BATCH_SIZE*cfg.GRAD_ACCUM} effective "
      f"| grad checkpointing {cfg.GRAD_CHECKPOINT} | visible GPUs {torch.cuda.device_count()}")
print(f"selection: on {cfg.SELECT_ON}, every {cfg.SAMPLE_EVERY} epochs after e{cfg.SEL_WARMUP_EPOCHS}, "
      f"{cfg.SEL_N_SAMPLE} masks at {cfg.SEL_DDIM_STEPS} DDIM steps | River gate "
      f"[{cfg.RIVER_GATE_LO}, {cfg.RIVER_GATE_HI}]x real | reset best on resume: "
      f"{cfg.RESET_BEST_ON_RESUME}")


## 3. Determinism — deliberately OFF here

This is the one runtime setting that differs from the segmentation notebooks, and it is not an
oversight. `torch.use_deterministic_algorithms(True)` routes part of this UNet's backward through a
sort-based kernel and costs roughly **6x wall-clock** on this workload. A diffusion run needs tens of
thousands of steps where a segmentation run needs a few thousand, so that factor is the difference
between finishing inside a session and not finishing at all.

Everything cheap is still pinned — Python/NumPy/torch RNG, worker seeding, the DataLoader generator,
and the fixed noise used by the validation monitor and the sampler. `cudnn.benchmark=True` is on
because every batch has identical shape. State the caveat in the paper: reproducible up to cuDNN
kernel selection, not bit-exact.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True          # fixed shapes -> autotune once, reuse
    torch.use_deterministic_algorithms(False)      # ~6x faster than True on this backward

def worker_init_fn(worker_id):
    s = cfg.SEED + worker_id
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
print("seeded:", cfg.SEED, "| deterministic algorithms: OFF | cudnn.benchmark:",
      torch.backends.cudnn.benchmark)

## 4. Filename parsing and base-tile grouping

Identical to the segmentation notebooks, and copied rather than rewritten for exactly that reason.
Convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the `mask`
prefix. `N` is ascending **export order with gaps**, and the three variants of a tile are *not*
adjacent in a sorted listing — so grouping is regex-based on `N`, never positional.

Stage 1 reads the masks only. The image paths are still parsed and asserted, because a dataset
version that renamed or dropped images would also invalidate the split Stage 2 depends on, and this
cell is where that should fail loudly.

In [ ]:
def find_data_root(path):
    if (Path(path) / "images").is_dir(): return str(path)
    hits = [p for p in Path("/kaggle/input").glob("**/satellite_dataset") if (p / "images").is_dir()]
    assert hits, "satellite_dataset/ not found under /kaggle/input -- attach sugam24/satellite-datasets"
    print(f"DATA_ROOT auto-detected: {hits[0]}")
    return str(hits[0])

DATA_ROOT = find_data_root(cfg.DATA_ROOT)
IMG_DIR  = Path(DATA_ROOT) / "images"
MASK_DIR = Path(DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axis
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

gaps = [b - a for a, b in zip(BASE_IDS, BASE_IDS[1:]) if b - a > 1]
print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print(f"{len(gaps)} gaps in export sequence (dropped tiles), total {sum(g-1 for g in gaps)} missing ids")
h, w = cv2.imread(str(MASK_DIR / records[0][3]), cv2.IMREAD_COLOR).shape[:2]
print(f"native mask size: {w}x{h}")

## 5. Mask colour diagnostic — run this before anything else

Same guard as the segmentation notebooks, and it matters more here, not less: the generator learns
whatever distribution the LUT hands it. If Residential and River were swapped by a channel-order
mistake, every sample, every KL, and every connectivity number below would still look plausible.

Two things this cell reports:
- the **orientation guard** — Residential must dominate River by a wide margin, else the BGR/RGB
  order is inverted;
- the **unmapped share** — the fraction of pixels whose colour is in neither the 6-colour table nor
  the ignore colour. This is required to read **0.000000%**. Anything else means CVAT polygon edge
  artifacts or resampling residue are silently being folded into ignore, which for a *generative*
  model means training on a class that does not exist.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}
IGNORE_BGR = (0, 0, 0)

# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.
def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

def _k(bgr): return int(pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0])

LUT   = np.full(1 << 24, cfg.IGNORE_INDEX, dtype=np.uint8)   # 16 MB; default = ignore
KNOWN = np.zeros(1 << 24, dtype=bool)                        # 16 MB; True only for palette colours
for bgr, idx in BGR_TO_CLASS.items():
    LUT[_k(bgr)] = idx; KNOWN[_k(bgr)] = True
KNOWN[_k(IGNORE_BGR)] = True

def rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:
    return LUT[pack_bgr(bgr_img)]

# ---- diagnostic over a random sample of raw full-res masks ----
rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
colour_hits, class_px, unmapped, total = {}, np.zeros(7, dtype=np.int64), 0, 0

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    keys = pack_bgr(m)
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = LUT[keys]
    class_px += np.bincount(np.where(idx == cfg.IGNORE_INDEX, 6, idx).ravel(), minlength=7)
    unmapped += int((~KNOWN[keys]).sum()); total += keys.size

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag_c = (f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known
             else ("-> IGNORE (palette black)" if c == IGNORE_BGR else "-> UNMAPPED -> ignore"))
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag_c}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(CODE_NAMES):
    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")
print(f"\nunmapped pixel share: {100*unmapped/total:.6f}%")

# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.
assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before training.")
assert unmapped == 0, f"{unmapped} pixels ({100*unmapped/total:.6f}%) are off-palette -- fix the masks first"
print("channel-order check PASSED | LUT is exact")

## 6. One-time repack to a class-index cache (128 and 512)

Same repack as the segmentation notebooks, minus the images — stage 1 never reads an image. Two
sizes are written: **128** is what the diffusion model trains on, **512** is the resolution of the
Stage-2 contract and is kept so that the real and generated masks can be compared at the same scale
Stage 2 will actually consume.

The ordering is the part that is easy to get wrong and is worth restating: colour -> class index at
**full 2345x2351 resolution first**, then `INTER_NEAREST` down. Resizing the RGB mask first would
interpolate between class colours and manufacture phantom classes at every polygon edge — and a
generative model would then learn those artifacts as if they were real geography.

Masks compress far better than images, so the whole cache is a few tens of MB rather than the
segmentation notebook's ~700 MB. **Run once, then `Save Version` -> add the output as a dataset ->
set `PREBUILT_CACHE`** if you want later sessions to skip it.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def find_cache():
    if cfg.PREBUILT_CACHE: return Path(cfg.PREBUILT_CACHE)
    hits = list(Path("/kaggle/input").glob("**/cache_idx/_complete.json"))
    if hits:
        print(f"prebuilt cache found: {hits[0].parent}")
        return hits[0].parent
    return Path(cfg.CACHE_DIR)

CACHE  = find_cache()
MARKER = CACHE / "_complete.json"
SIZES  = (cfg.GEN_SIZE, cfg.OUT_SIZE)

def _repack_one(rec):
    _, _, _, mname = rec
    msk  = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    keys = pack_bgr(msk)
    lab  = LUT[keys]                                            # full-res -> class index
    bad  = int((~KNOWN[keys]).sum())
    for s in SIZES:
        small = cv2.resize(lab, (s, s), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(str(CACHE / f"labels{s}" / mname), small, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == cfg.IGNORE_INDEX, 6, lab).ravel(), minlength=7), bad, lab.size

if MARKER.exists():
    meta = json.load(open(MARKER))
    print(f"cache present at {CACHE} ({meta['n']} masks) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    print("building the index cache -- about 12 min, once per session unless you save it as a dataset")
    for s in SIZES:
        (CACHE / f"labels{s}").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts, bad_px, tot_px = {}, 0, 0
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, (b, bad, n) in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[3]] = b.tolist(); bad_px += bad; tot_px += n
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts, "sizes": list(SIZES),
               "unmapped_px": bad_px, "total_px": tot_px,
               "unmapped_pct": 100.0 * bad_px / max(tot_px, 1)}, open(MARKER, "w"))
    print(f"repacked {len(counts)} masks in {time.time()-t0:.0f}s")

meta = json.load(open(MARKER))
PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in meta["counts"].items()}
print(f"LUT unmapped over the FULL dataset: {meta.get('unmapped_pct', float('nan')):.6f}% "
      f"({meta.get('unmapped_px', 0)} px)")
assert meta.get("unmapped_pct", 0.0) == 0.0, "off-palette pixels present in the full dataset"
for s in SIZES:
    n = len(list((CACHE / f"labels{s}").glob("*.png")))
    print(f"  labels{s}: {n} files")
sz = sum(f.stat().st_size for f in CACHE.rglob('*.png')) / 1e6
print(f"cache size: {sz:.1f} MB")

## 7. Load the 128 cache into RAM

1224 x 128 x 128 uint8 = **20 MB**. There is no reason not to hold it resident: it removes disk I/O
from the training loop entirely, and at 25 steps per epoch the loader would otherwise be a
noticeable fraction of the epoch. The 512 cache stays on disk — it is only read once, for the
real-vs-generated comparison, and never inside the training loop.

The assert is the same one as the segmentation notebooks: labels must contain nothing but
`{0..5, 255}`. For a generative model an unexpected value would not raise a device-side assert, it
would just quietly become a seventh class.

In [ ]:
RAM = {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, _, mname) in enumerate(records):
        RAM[mname] = cv2.imread(str(CACHE / f"labels{cfg.GEN_SIZE}" / mname), cv2.IMREAD_GRAYSCALE)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = sum(a.nbytes for a in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} masks, {mb:.0f} MB, {time.time()-t0:.0f}s")
    u = np.unique(np.concatenate([a[::8, ::8].ravel() for a in RAM.values()]))
    assert set(u.tolist()) <= set(range(cfg.N_CLASSES)) | {cfg.IGNORE_INDEX}, \
        f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())

GLOBAL_PX = np.sum(list(PIX_COUNTS.values()), axis=0)
print("full-dataset class share (%):",
      {CODE_NAMES[i]: round(100 * GLOBAL_PX[i] / GLOBAL_PX.sum(), 2) for i in range(7)})

## 8. Splitter — fold 1 of the sequence-block CV

`block_split` is **copied verbatim** from the segmentation notebooks. It must be, because Stage 2
runs the identical function and the two stages have to agree on which tiles exist. Do not
reimplement it, do not tidy it, do not change the buffer.

For **fold 1** that resolves to: sort the 408 base ids ascending, cut into 3 contiguous blocks,
block 1 (positions 136..271) is validation, blocks 0+2 are train, and the `BLOCK_BUFFER=2` tiles on
each side of the validation block are dropped from **train** — never from val. Result:
**268 train tiles / 136 val tiles / 4 buffered out**, or 804 train masks once the three flip
variants of each train tile are pulled in. Fold 1 buffers 4 tiles rather than fold 0's 2 because an
interior validation block has two boundaries with train, not one.

### The leakage rule
The generator is fitted on `tr_recs` and nothing else. `va_recs` is used in exactly one place — the
epoch-level eps-MSE monitor in section 17 — which computes a loss and never a gradient. Set
`VAL_MONITOR=False` if you would rather the val tiles were not touched at all; the cost is that
model selection falls back to the last epoch instead of the best one.

The split is written to `fold{FOLD}_split.json` with a SHA1 of the train id list, so a one-line diff
against Stage 2's copy settles any argument about whether the two notebooks agree.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

assert cfg.SPLIT_MODE == "block", "stage 1 is defined on the block split"
tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)

def expand(ids): return [r for i in ids for r in groups[i]]
tr_recs, va_recs = expand(tr_ids), expand(va_ids)

assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"
buffered = sorted(set(BASE_IDS) - set(tr_ids) - set(va_ids))
split_sha1 = hashlib.sha1(",".join(map(str, sorted(tr_ids))).encode()).hexdigest()

SPLIT = {"split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "n_folds": cfg.N_FOLDS,
         "block_buffer": cfg.BLOCK_BUFFER, "rule": "block_split(), verbatim from the part-1 notebooks",
         "train_ids": sorted(tr_ids), "val_ids": sorted(va_ids), "buffered_ids": buffered,
         "train_sha1": split_sha1}
SPLIT_NAME = f"fold{cfg.FOLD}_split.json"
json.dump(SPLIT, open(f"{cfg.OUT_DIR}/{SPLIT_NAME}", "w"), indent=2)

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "
      f"({len(buffered)} buffered out: {buffered})")
print(f"  -> {len(tr_recs)} train masks / {len(va_recs)} val masks")
print(f"  val id range: {min(va_ids)}..{max(va_ids)} | train id range: {min(tr_ids)}..{max(tr_ids)}")
print(f"  train_sha1: {split_sha1}   <-- must match Stage 2")
print(f"  saved -> {cfg.OUT_DIR}/{SPLIT_NAME}")

## 9. One-hot label coding — and why the binary code failed

This replaces the loss section of the segmentation notebooks, and it is the single most
consequential modelling choice in stage 1. Run A used the SatSynth binary code; this run uses
one-hot. Both are kept, because the comparison is a result.

### What run A did
`ceil(log2 7) = 3` channels, class index written in binary, decoded by thresholding each channel
at zero:

| code | class | code | class |
|---|---|---|---|
| 000 | Residential | 100 | UnusedLand |
| 001 | Road | 101 | Agricultural |
| 010 | **River** | 110 | Ignore |
| 011 | Forest | 111 | illegal |

### Why that breaks at 7 classes
The three channels are denoised independently and thresholded independently, so nothing ties them
together at a class boundary. **River (010) is the exact midpoint between Residential (000) and
Forest (011)** — the most common boundary in the dataset, 52.6% of pixels meeting 12.4%. Wherever
channel 1 crosses zero a pixel before channel 2, the decoder emits River. Likewise Agricultural
(101) is the midpoint of Road (001) and UnusedLand (100), and Forest (011) meeting Agricultural
(101) has the illegal code 111 as its midpoint.

Measured on run A's epoch-699 grid against fold-1 train: River **7.7x** over, Forest **0.54x**,
Agricultural 1.62x, illegal-code rate 0.479%. Those River blobs are not learned rivers — they are
decoded boundary artefacts, which is why they fill 2D interfaces instead of tracing 1D channels.

### What one-hot changes
Seven channels, `+1` on the true class and `-1` elsewhere, decoded by `argmax`. A small
per-channel error can only move the argmax between the two channels that are already competing at
that pixel — i.e. between the two classes genuinely present at the boundary. **A third, unrelated
class can no longer appear, and there is no illegal code at all**, so the nearest-legal-code
projection disappears with it.

The cost is four extra input/output channels: at `BASE_CH=64` that is under 0.1% of the parameters.
SatSynth's channel saving is worth having at 100+ classes; at 7 it buys nothing and costs the
boundaries.

### The diagnostic that replaces `illegal_code_rate`
`decode_codes` now reports an **ambiguity rate**: the share of pixels where the top two channels
are within `AMB_MARGIN`. It plays the same role — a rising ambiguity rate means samples are not
settling on the code corners — but unlike the illegal rate it cannot be zero while the decode is
still wrong, because there is nothing left to project.


In [ ]:
ONEHOT = np.eye(cfg.N_CODES, dtype=np.float32) * 2.0 - 1.0      # (7, 7), rows in {-1, +1}

def encode_index(lab: np.ndarray) -> np.ndarray:
    # (H, W) indices 0..6 -> (7, H, W) float32: +1 on the true class, -1 on the other six
    return np.ascontiguousarray(ONEHOT[lab].transpose(2, 0, 1))

@torch.no_grad()
def decode_codes(x: torch.Tensor, margin: Optional[float] = None):
    # (B, 7, H, W) continuous -> (B, H, W) long indices 0..6, plus the AMBIGUITY rate.
    #
    # argmax over the 7 channels cannot name a class outside 0..6, so unlike the binary code there
    # is no illegal state and no nearest-legal-code projection. What can still go wrong is a pixel
    # where two channels are nearly tied; the rate below counts those, and it is the series to
    # watch in W&B in place of run A's illegal_code_rate.
    margin = cfg.AMB_MARGIN if margin is None else margin
    x = x.clamp(-1, 1)
    top2 = torch.topk(x, 2, dim=1)
    idx  = top2.indices[:, 0].long()
    rate = float(((top2.values[:, 0] - top2.values[:, 1]) < margin).float().mean())
    return idx, rate

# ---- round trip: clean codes must be exact, noisy codes must degrade gracefully ----
_a = np.random.RandomState(cfg.SEED).randint(0, cfg.N_CODES, (4, 64, 64)).astype(np.uint8)
_x = torch.from_numpy(np.stack([encode_index(m) for m in _a]))
_d, _r = decode_codes(_x)
assert (_d.numpy() == _a).all() and _r == 0.0, "one-hot round trip is not exact"
print("codes (index -> channels):")
for i, c in enumerate(ONEHOT.astype(int)):
    print(f"  {i} {CODE_NAMES[i]:<14} {c.tolist()}")

# Side-by-side degradation against the binary code that run A used. This cell is the source of the
# ablation-A-vs-B number in the paper, so it runs every time rather than living in a scratch file.
_BIN = np.array([[(i >> 2) & 1, (i >> 1) & 1, i & 1] for i in range(cfg.N_CODES)],
                dtype=np.float32) * 2.0 - 1.0
print(f"\n{'sigma':>6} | {'onehot acc':>10} {'amb rate':>9} | {'binary acc':>10} {'illegal':>9}")
for _s in (0.30, 0.45, 0.60):
    _n = torch.randn_like(_x) * _s
    _do, _ro = decode_codes(_x + _n)
    _xb = torch.from_numpy(np.stack([_BIN[m].transpose(2, 0, 1) for m in _a]))
    _bb = ((_xb + torch.randn_like(_xb) * _s) > 0).to(torch.uint8)
    _ib = (_bb[:, 0] * 4 + _bb[:, 1] * 2 + _bb[:, 2]).numpy()
    print(f"{_s:6.2f} | {(_do.numpy()==_a).mean():10.4f} {_ro:9.4f} | "
          f"{(_ib==_a).mean():10.4f} {(_ib==7).mean():9.4f}")
print("\nsame noise, same task: the binary decode loses pixels to classes that are not at the "
      "boundary and to code 7; the one-hot decode cannot do either.")


## 10. Dataset and DataLoaders

**No hflip/vflip** — those are already materialised in the 1224 files, exactly as in the
segmentation notebooks. Adding them would produce duplicate views inside a single epoch.

On-the-fly augmentation is **rot90 (k in 0..3) only**. Combined with the baked-in flips this
completes the dihedral group D4, the natural symmetry group for nadir overhead imagery, and rot90 is
exact — no resampling, so 9 px roads and 13 px rivers stay crisp. The photometric jitter from the
segmentation notebooks has no meaning here: there is no image, only a label field.

Augmentation is seeded per (epoch, index) so the run is reproducible and workers never collide. The
dataset yields the encoded tensor directly; there is no target, because in an unconditional DDPM the
target is the noise, drawn fresh inside the training step.

810 train images at batch 32 is **25 steps per epoch**, which is small — that is why `MAX_EPOCHS` is
1500 rather than 200. Think in steps (~38 k), not epochs.

In [ ]:
class MaskCodes(Dataset):
    def __init__(self, recs, train: bool):
        self.recs, self.train, self.epoch = recs, train, 0

    def __len__(self): return len(self.recs)

    def _read(self, mname):
        if cfg.PRELOAD_RAM: return RAM[mname].copy()
        return cv2.imread(str(CACHE / f"labels{cfg.GEN_SIZE}" / mname), cv2.IMREAD_GRAYSCALE)

    def __getitem__(self, i):
        _, _, _, mname = self.recs[i]
        lab = self._read(mname)
        if self.train and cfg.AUG_ROT90:
            rs = np.random.RandomState((cfg.SEED * 100003 + self.epoch * 9973 + i) % (2**31))
            k = rs.randint(4)
            if k: lab = np.ascontiguousarray(np.rot90(lab, k))
        lab = np.where(lab == cfg.IGNORE_INDEX, cfg.IGNORE_CODE, lab).astype(np.uint8)  # 255 -> 6
        return torch.from_numpy(encode_index(lab))

train_ds, val_ds = MaskCodes(tr_recs, True), MaskCodes(va_recs, False)

# Optional River oversampling. OFF by default -- see the CFG comment. When on it changes the
# distribution the model is trained on, so it is recorded in ARCH_SIG and on the model card.
train_sampler, _shuffle = None, True
if cfg.RIVER_OVERSAMPLE > 0:
    from torch.utils.data import WeightedRandomSampler
    _has_river = np.array([PIX_COUNTS[r[3]][2] > 0 for r in tr_recs])
    _w = np.where(_has_river, 1.0 + cfg.RIVER_OVERSAMPLE, 1.0)
    train_sampler = WeightedRandomSampler(torch.as_tensor(_w, dtype=torch.double),
                                          num_samples=len(tr_recs), replacement=True, generator=g)
    _shuffle = False
    print(f"River oversampling ON: {_has_river.sum()}/{len(tr_recs)} tiles contain River, "
          f"weight x{1.0 + cfg.RIVER_OVERSAMPLE:.2f} -- this biases the learned marginal")

train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=_shuffle, sampler=train_sampler,
                      drop_last=True, num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      persistent_workers=True, worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn)

xb = next(iter(train_ld))
_dec, _rate = decode_codes(xb)
assert xb.shape[1] == cfg.N_CH, f"encoder emitted {xb.shape[1]} channels, expected {cfg.N_CH}"
assert _rate == 0.0 and set(torch.unique(xb).tolist()) <= {-1.0, 1.0}, "encoder emitted non-code values"
assert (xb.max(dim=1).values == 1.0).all() and (xb == 1.0).sum() == xb[:, 0].numel(), \
    "one-hot invariant broken: every pixel must have exactly one +1 channel"
print(xb.shape, xb.dtype, "| values:", sorted(set(torch.unique(xb).tolist())),
      "| decoded codes:", sorted(torch.unique(_dec).tolist()))
print(f"{len(train_ds)} train masks / {len(val_ds)} val masks | "
      f"{len(train_ld)} steps per epoch | ~{len(train_ld)*cfg.MAX_EPOCHS/1000:.0f} k steps planned")

TRAIN_128 = np.stack([np.where(RAM[r[3]] == cfg.IGNORE_INDEX, cfg.IGNORE_CODE, RAM[r[3]])
                      for r in tr_recs]).astype(np.uint8)      # real reference set for section 13
VAL_128   = np.stack([np.where(RAM[r[3]] == cfg.IGNORE_INDEX, cfg.IGNORE_CODE, RAM[r[3]])
                      for r in va_recs]).astype(np.uint8)
print("real arrays:", TRAIN_128.shape, VAL_128.shape)
_p = np.bincount(TRAIN_128.ravel(), minlength=cfg.N_CODES) / TRAIN_128.size
print("fold-%d TRAIN class share (%%): " % cfg.FOLD +
      ", ".join(f"{CODE_NAMES[i]} {100*_p[i]:.2f}" for i in range(cfg.N_CODES)))
print("  ^ this, not the whole-dataset share, is what the generator is asked to reproduce")
# Real baselines for the two monitors that have no target in the training log. Without these,
# "masks with River 0.50" is uninterpretable: it may be exactly right or a total collapse.
_wr = float(np.mean([(a == 2).any() for a in TRAIN_128]))
_wd = float(np.mean([(a == 1).any() for a in TRAIN_128]))
_ss = [np.bincount(a.ravel(), minlength=cfg.N_CODES) / a.size for a in TRAIN_128]
_shdi = float(np.mean([-(q[q > 0] * np.log(q[q > 0])).sum() for q in _ss]))
print(f"real frac_with_River {_wr:.3f} | real frac_with_Road {_wd:.3f} | real SHDI {_shdi:.4f}")
print("  ^ compare these against sel_frac_with_River / sel_shdi in the loop below")


## 11. Forward diffusion process

`T=1000`, linear beta from 1e-4 to 2e-2, **v-parameterisation**, zero terminal SNR. The forward
step is `x_t = sqrt(acp_t) x_0 + sqrt(1-acp_t) eps` and the model is trained to predict
`v = sqrt(acp_t) eps - sqrt(1-acp_t) x_0`.

### Why not epsilon
Run A combined `ZERO_TERMINAL_SNR=True` with `prediction_type="epsilon"`. Those are incompatible.
Rescaling the betas forces `alpha_bar_T = 0`, so at `t = T` the input is pure noise and the epsilon
target *is* the input — the model learns the identity there and the first sampling steps carry no
information about the global mean. The symptom is a flattened marginal: run A trained on 52.61%
Residential and generated 42.99%, with every rare class inflated and Shannon diversity up. Lin et
al. (WACV 2024), the same paper the zero-SNR rescale comes from, prescribe v-prediction for
precisely this case. The assert below refuses the broken pair rather than letting it train for
20 hours.

Three details worth stating because they are the ones that silently break a diffusion run:
- **The training schedule and the sampling schedule must be the same object mathematically.**
  Training uses the tensors built here; sampling uses `DDIMScheduler` built from the same four
  numbers. The assert compares the two `alpha_bar` curves elementwise.
- **`clip_sample=True` with range 1.0** is correct here. The data lives on the corners of
  `[-1,1]^7`, so clipping the predicted `x_0` to that cube at every DDIM step is a genuine prior.
- **Loss values are not comparable to run A.** v-MSE and eps-MSE are different quantities on
  different scales. Compare the two runs on class-share KL, connectivity and downstream mIoU —
  never on the training curve.


In [ ]:
from diffusers import UNet2DModel, DDIMScheduler

class DDPMSchedule:
    def __init__(self, T, b0, b1, device):
        self.T = T
        self.betas = torch.linspace(b0, b1, T, dtype=torch.float32, device=device)
        self.alphas = 1.0 - self.betas
        self.acp = torch.cumprod(self.alphas, dim=0)
        self.sqrt_acp = self.acp.sqrt()
        self.sqrt_1m_acp = (1.0 - self.acp).sqrt()

    def q_sample(self, x0, t, noise):
        return (self.sqrt_acp[t].view(-1, 1, 1, 1) * x0 +
                self.sqrt_1m_acp[t].view(-1, 1, 1, 1) * noise)

    def v_target(self, x0, t, noise):
        # v = sqrt(acp) eps - sqrt(1-acp) x0   (Salimans & Ho, 2022)
        return (self.sqrt_acp[t].view(-1, 1, 1, 1) * noise -
                self.sqrt_1m_acp[t].view(-1, 1, 1, 1) * x0)

    def target(self, x0, t, noise):
        return self.v_target(x0, t, noise) if cfg.PREDICTION_TYPE == "v_prediction" else noise

    def pred_to_x0(self, xt, t, pred):
        a = self.sqrt_acp[t].view(-1, 1, 1, 1)
        s = self.sqrt_1m_acp[t].view(-1, 1, 1, 1)
        if cfg.PREDICTION_TYPE == "v_prediction":
            return a * xt - s * pred
        return (xt - s * pred) / a.clamp(min=1e-8)      # epsilon: undefined at alpha_bar_T = 0

def rescale_zero_terminal_snr(betas):
    # Lin et al. 2024. A linear schedule leaves alpha_bar_T ~ 4e-5, not 0, so ~0.6% of the signal
    # survives to t=T. Sampling starts from pure zero-mean noise, so the model can never reproduce
    # a strong global offset -- it systematically under-produces the dominant class.
    acp = torch.cumprod(1.0 - betas, dim=0)
    sa = acp.sqrt()
    sa0, saT = sa[0].clone(), sa[-1].clone()
    sa = (sa - saT) * (sa0 / (sa0 - saT))          # force sqrt(alpha_bar_T) = 0
    acp = sa ** 2
    alphas = torch.cat([acp[0:1], acp[1:] / acp[:-1]])
    return 1.0 - alphas

class _ZSNRSchedule(DDPMSchedule):
    def __init__(self, T, b0, b1, device):
        super().__init__(T, b0, b1, device)
        self.betas = rescale_zero_terminal_snr(self.betas)
        self.alphas = 1.0 - self.betas
        self.acp = torch.cumprod(self.alphas, dim=0).clamp(min=0.0)
        self.sqrt_acp = self.acp.sqrt()
        self.sqrt_1m_acp = (1.0 - self.acp).clamp(min=0.0).sqrt()

# The guard that run A did not have. With alpha_bar_T = 0 the epsilon target equals the network
# input at t=T, so the model cannot learn the global mean and the class marginals come out
# flattened. Either use v-prediction, or turn the zero-SNR rescale off -- not neither.
assert not (cfg.ZERO_TERMINAL_SNR and cfg.PREDICTION_TYPE == "epsilon"), (
    "ZERO_TERMINAL_SNR=True with prediction_type='epsilon' is degenerate at t=T. "
    "Set PREDICTION_TYPE='v_prediction' (recommended) or ZERO_TERMINAL_SNR=False.")

_S = _ZSNRSchedule if cfg.ZERO_TERMINAL_SNR else DDPMSchedule
sched = _S(cfg.TIMESTEPS, cfg.BETA_START, cfg.BETA_END, "cuda")

def build_ddim():
    kw = dict(num_train_timesteps=cfg.TIMESTEPS, beta_start=cfg.BETA_START, beta_end=cfg.BETA_END,
              beta_schedule="linear", prediction_type=cfg.PREDICTION_TYPE,
              clip_sample=True, set_alpha_to_one=False, steps_offset=0)
    if cfg.ZERO_TERMINAL_SNR:
        # trailing spacing puts a step exactly at t=T, which is the point of zeroing the SNR
        kw.update(rescale_betas_zero_snr=True, timestep_spacing="trailing", set_alpha_to_one=True)
    for extra in ({"clip_sample_range": 1.0}, {}):       # older diffusers lacks some kwargs
        try:    return DDIMScheduler(**kw, **extra)
        except TypeError: continue
    kw.pop("rescale_betas_zero_snr", None); kw.pop("timestep_spacing", None)
    return DDIMScheduler(**kw)

ddim_probe = build_ddim()
assert torch.allclose(ddim_probe.alphas_cumprod.cuda().float(), sched.acp, atol=1e-4), \
    "DDIM schedule does not match the training schedule"

# Round trip on the parameterisation itself: v -> x0 must recover x0 exactly at every t.
_x0 = torch.from_numpy(encode_index(
    np.random.RandomState(0).randint(0, cfg.N_CODES, (32, 32)).astype(np.uint8)))[None].cuda()
for _t in (0, 250, 500, 750, 999):
    _tt = torch.tensor([_t], device="cuda")
    _e = torch.randn_like(_x0)
    _xt = sched.q_sample(_x0, _tt, _e)
    _v = sched.target(_x0, _tt, _e)
    assert torch.allclose(sched.pred_to_x0(_xt, _tt, _v), _x0, atol=1e-3), f"v->x0 broken at t={_t}"
print("v-parameterisation round trip exact at t = 0/250/500/750/999")

print(f"zero terminal SNR: {cfg.ZERO_TERMINAL_SNR} | alpha_bar_T = {float(sched.acp[-1]):.3e} "
      f"| prediction {cfg.PREDICTION_TYPE} | offset noise {cfg.OFFSET_NOISE}")
print(f"beta {sched.betas[0]:.2e} -> {sched.betas[-1]:.2e} | "
      f"alpha_bar {sched.acp[0]:.5f} -> {sched.acp[-1]:.2e}")
print(f"train/sample schedules agree | DDIM steps {cfg.DDIM_STEPS}, eta {cfg.DDIM_ETA}")
_snr = (sched.acp / (1 - sched.acp).clamp(min=1e-12)).sqrt()
print("SNR at t=0/250/500/750/999:", [f"{float(_snr[i]):.2f}" for i in (0, 250, 500, 750, 999)])


## 12. Model, optimizer, scheduler, EMA

### Architecture
`UNet2DModel` with `block_out_channels=(128, 256, 256, 512)` over resolutions 128/64/32/16, two
residual blocks per level, and self-attention at 32 and 16. That is the standard DDPM topology for
this resolution and lands in the **30-60 M** band; the exact count is printed below and stamped into
`ARCH_SIG`, which every checkpoint carries and which `load` refuses to override.

In and out channels are both 3 — the binary code, not RGB. If you ever ablate one-hot, this is the
one number that changes here (`in_channels=out_channels=7`), plus the coding block in section 9.

### Parameter groups
| Group | LR | Weight decay |
|---|---|---|
| conv/linear weights | 2e-4 | 0.0 |
| norm gains + biases | 2e-4 | 0.0 |

Diffusion UNets are conventionally trained without weight decay, so `WEIGHT_DECAY=0.0` makes the
split a no-op — the group structure is kept anyway so that turning decay on is a config change and
not a code change, and so norm gains are never decayed if you do.

### Schedule
500 linear warmup steps then cosine decay to zero at `COS_HORIZON_EPOCHS`. Unlike the segmentation
runs there is no early stopping to fire first, so the horizon is set equal to `MAX_EPOCHS`: the run
is meant to reach the end of the cosine.

### EMA
Non-optional for diffusion, which is why `USE_EMA` defaults to True and every sample in this
notebook comes from `ema.ema`. Raw weights at the end of a diffusion run produce visibly worse
samples than their moving average; `decay=0.9995` over ~38 k steps averages roughly the last 2000.
The decay ramp on the first few hundred steps is the same one the segmentation notebooks use.

### Memory — why batch 32 does not fit on one T4, and what is done about it
A UNet at 128x128 stores far more activation memory than a segmentation encoder at 512, because the
**up-blocks concatenate the skip connections at full resolution**: the top level runs three residual
blocks over a 256-channel 128x128 tensor per sample, and the whole chain is retained for backward.
At batch 32 that is roughly 13 GB and a T4 has 14.6 GB, so it OOMs inside the first up-block.

Three levers, applied in this order, none of which changes the protocol:

| Lever | Effect | Cost |
|---|---|---|
| `BATCH_SIZE=16`, `GRAD_ACCUM=2` | halves resident activations; **effective batch stays 32** | none — the gradient is identical up to fp16 rounding |
| `GRAD_CHECKPOINT=True` | ~3x less activation memory (blocks are recomputed in backward) | ~30% slower per step |
| `MULTI_GPU=True` | splits each micro-batch across both T4s | needs the accelerator set to **T4 x2**; ~1.6-1.8x faster |

`expandable_segments:True` is set in section 1 as well — it does not reduce demand, it just stops
fragmentation from turning a 13.5 GB requirement into a 14.6 GB failure.

**Effective batch is protocol, micro-batch is not.** If you still OOM, halve `BATCH_SIZE` and double
`GRAD_ACCUM` again (8 x 4); do not simply lower `BATCH_SIZE` on its own, because that changes the
optimization and makes this run incomparable to any rerun.

### Multi-GPU — opt-in, and not needed for memory
`MULTI_GPU` defaults to **False**, because the two levers above already fit the run on a single T4
with room to spare. Turn it on only for the ~1.6-1.8x speedup, and only if you have set the
accelerator to **T4 x2**.

`nn.DataParallel` wraps a thin adapter (`_Fwd`) rather than the UNet itself, for two reasons: it must
return a plain tensor, since `UNet2DOutput` does not survive DataParallel's gather; and the timestep
has to be a per-sample tensor of shape `(B,)` so it scatters along the batch dimension alongside the
input. `model` stays the **raw, unwrapped module** — optimizer, EMA, checkpoints and the sampler all
use it, so nothing downstream has to know whether DataParallel is on, and a checkpoint written on
two GPUs loads on one.

There is also a shim. `UNet2DModel.forward` asks for `self.dtype`, which diffusers answers by walking
the module's registered parameters — and a DataParallel *replica* has none, because `replicate()`
stores the copied tensors in each submodule's `__dict__` instead. In this diffusers version the
fallback branch for that case raises `UnboundLocalError` rather than returning a dtype, so the
property is replaced with a replica-safe version. It is applied only when DataParallel is actually
in use, so a single-GPU run touches no library internals at all.

This is the general shape of the trade: DataParallel is a 30-year-old-style thread-per-GPU design
that library code does not always expect, so it buys throughput at the cost of a small integration
risk. The backward guard at the end of this cell is what makes that risk cheap — it fails here, in
five seconds, rather than three hours into a run that was silently training nothing.

In [ ]:
def build_model():
    chans = tuple(cfg.BASE_CH * m for m in cfg.CH_MULT)
    res   = [cfg.GEN_SIZE // (2 ** i) for i in range(len(chans))]
    down  = tuple("AttnDownBlock2D" if r in cfg.ATTN_RES else "DownBlock2D" for r in res)
    up    = tuple("AttnUpBlock2D" if r in cfg.ATTN_RES else "UpBlock2D" for r in reversed(res))
    return UNet2DModel(
        sample_size=cfg.GEN_SIZE,
        in_channels=cfg.N_CH, out_channels=cfg.N_CH,         # one-hot label channels, not RGB
        layers_per_block=cfg.LAYERS_PER_BLOCK,
        block_out_channels=chans,
        down_block_types=down, up_block_types=up,
        attention_head_dim=cfg.ATTN_HEAD_DIM,
        norm_num_groups=cfg.NORM_GROUPS,
    )

model = build_model().cuda()                      # raw module: optimizer, EMA, checkpoints, sampler
if cfg.GRAD_CHECKPOINT:
    model.enable_gradient_checkpointing()         # recompute blocks in backward instead of storing

class _Fwd(nn.Module):
    # DataParallel needs a tensor return, and a timestep that scatters along the batch dim
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x, t): return self.m(x, t).sample

N_GPU  = torch.cuda.device_count()
USE_DP = bool(cfg.MULTI_GPU and N_GPU > 1)

if USE_DP:
    # DataParallel shim. UNet2DModel.forward calls `self.dtype`, which diffusers resolves by walking
    # registered parameters. A DataParallel *replica* has none registered -- replicate() stores the
    # copied tensors in __dict__ instead -- so it falls into a fallback branch that raises
    # UnboundLocalError in this version. Resolve the dtype ourselves, replica-safe.
    from diffusers.models.modeling_utils import ModelMixin
    def _dp_safe_dtype(self):
        for t in itertools.chain(self.parameters(), self.buffers()):
            return t.dtype
        for m in self.modules():                    # replica: tensors live in each module's __dict__
            for v in m.__dict__.values():
                if torch.is_tensor(v): return v.dtype
        return torch.float32
    ModelMixin.dtype = property(_dp_safe_dtype)
    print("DataParallel dtype shim applied")

net = nn.DataParallel(_Fwd(model)) if USE_DP else _Fwd(model)   # training forward goes through net

def param_groups(m):
    dec, nod = [], []
    for n, p in m.named_parameters():
        if not p.requires_grad: continue
        (nod if (p.ndim <= 1 or n.endswith(".bias")) else dec).append(p)
    wd  = cfg.WEIGHT_DECAY
    nwd = 0.0 if cfg.NO_DECAY_ON_NORM_BIAS else wd
    return [{"params": dec, "lr": cfg.LR, "weight_decay": wd,  "name": "decay"},
            {"params": nod, "lr": cfg.LR, "weight_decay": nwd, "name": "nodecay"}]

optimizer = torch.optim.AdamW(param_groups(model), weight_decay=cfg.WEIGHT_DECAY, betas=(0.9, 0.999))

steps_per_epoch = len(train_ld) // cfg.GRAD_ACCUM
total_steps = steps_per_epoch * cfg.COS_HORIZON_EPOCHS

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return (step + 1) / cfg.WARMUP_STEPS
    prog = (step - cfg.WARMUP_STEPS) / max(1, total_steps - cfg.WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)

class ModelEMA:
    def __init__(self, m, decay):
        self.ema = copy.deepcopy(m).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, m, step):
        # ramp the decay in, otherwise the average stays pinned to the init for thousands of steps
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        msd = m.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else: v.copy_(msd[k])

ema = ModelEMA(model, cfg.EMA_DECAY) if cfg.USE_EMA else None
eval_model = ema.ema if cfg.USE_EMA else model

n_par = sum(p.numel() for p in model.parameters())
ARCH_SIG = (f"unet2d|{cfg.GEN_SIZE}|{tuple(cfg.BASE_CH*m for m in cfg.CH_MULT)}|"
            f"lpb{cfg.LAYERS_PER_BLOCK}|attn{cfg.ATTN_RES}|{cfg.LABEL_CODING}{cfg.N_CH}|"
            f"T{cfg.TIMESTEPS}|{cfg.PREDICTION_TYPE}|zsnr{int(cfg.ZERO_TERMINAL_SNR)}|"
            f"off{cfg.OFFSET_NOISE}|ros{cfg.RIVER_OVERSAMPLE}|p{n_par}")
with torch.no_grad():
    probe = model(torch.zeros(2, cfg.N_CH, cfg.GEN_SIZE, cfg.GEN_SIZE, device="cuda"),
                  torch.tensor([0, cfg.TIMESTEPS - 1], device="cuda")).sample
assert probe.shape[1:] == (cfg.N_CH, cfg.GEN_SIZE, cfg.GEN_SIZE), probe.shape
assert 8e6 <= n_par <= 80e6, f"{n_par/1e6:.1f} M params is outside the intended band"

# Guard: checkpointing that detaches the graph would train nothing and never raise. Check that a
# real backward actually reaches the parameters, through whatever wrapper is in use.
model.train()
_x = torch.randn(2 * max(N_GPU, 1), cfg.N_CH, cfg.GEN_SIZE, cfg.GEN_SIZE, device="cuda")
_t = torch.randint(0, cfg.TIMESTEPS, (_x.shape[0],), device="cuda")
with torch.cuda.amp.autocast(enabled=cfg.AMP):
    _o = net(_x, _t)
F.mse_loss(_o.float(), torch.randn_like(_x)).backward()
_gn = sum(float(p.grad.norm()) for p in model.parameters() if p.grad is not None)
assert _gn > 0, "no gradient reached the parameters -- gradient checkpointing detached the graph"
model.zero_grad(set_to_none=True)
del _x, _t, _o; gc.collect(); torch.cuda.empty_cache()

print(f"params: {n_par/1e6:.2f} M | forward out: {tuple(probe.shape)} | backward guard OK")
print(f"steps/epoch: {steps_per_epoch} | cosine horizon: {total_steps} steps "
      f"({cfg.COS_HORIZON_EPOCHS} epochs) | EMA: {cfg.USE_EMA}")
print(f"memory plan: micro-batch {cfg.BATCH_SIZE} x accum {cfg.GRAD_ACCUM} "
      f"(effective {cfg.BATCH_SIZE*cfg.GRAD_ACCUM}) | checkpointing {cfg.GRAD_CHECKPOINT} | "
      f"DataParallel {USE_DP} over {N_GPU} GPU(s)")
print("arch:", ARCH_SIG)


## 13. Mask-realism metrics and rendering

This replaces the segmentation notebook's confusion-matrix module. There is no ground truth to
compare a *generated* mask against, so realism is measured as a distribution match between the
generated set and the real fold-1 train set, at 128 for both.

- **Class proportions + KL.** Mean per-code pixel share, and `KL(real||gen)` over the 7-bin
  distribution. Two failure modes: a model that quietly drops River, and
  run A's, which invented 7.7x too much of it at class boundaries. KL catches both; the one-sided
  checks in run A caught neither.
- **Landscape metrics**, per class: patch density (patches per image, 8-connectivity), mean patch
  area, edge density (class-boundary pixel pairs per 1000 px of image), and largest-patch share.
  Shannon diversity is a *landscape*-level quantity by definition, so it is computed per image over
  the class proportions and summarised as mean +/- sd rather than reported per class.
- **Connectivity**, for Road and River only. `frag` = connected components per 1000 class pixels. A
  real river is a handful of long components, so `frag` is small; a generator that sprays
  disconnected dots pushes it up while leaving the *class share* looking perfectly healthy. That is
  precisely the failure that a class-proportion histogram cannot see, and it is why this number
  exists.
- **Nearest-neighbour memorization check.** Minimum per-pixel disagreement between each generated
  mask and every train mask, at 32x32. The baseline is the same statistic computed for *held-out
  val* masks against train: real unseen tiles sit at some natural distance from the training set,
  and generated masks should sit at a comparable distance. If the generated distances collapse
  toward zero, the model is reciting tiles and the whole augmentation argument fails.

The palette block is the same BGR-table-reversed-to-RGB used in the segmentation notebooks, so the
figures in this notebook and in those are directly comparable.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import ndimage as ndi

STRUCT8 = np.ones((3, 3), dtype=int)          # 8-connectivity: correct for linear features

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB for matplotlib
PALETTE_RGB[cfg.IGNORE_CODE] = (0, 0, 0)               # code 6, inside the model
PALETTE_RGB[cfg.IGNORE_INDEX] = (0, 0, 0)              # 255, on disk

def colorize(lab): return PALETTE_RGB[lab]

LEGEND = ([Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)] +
          [Patch(color="black", label="ignore/unannotated")])

def mask_grid(stack, ncol=4, title=None, cell=2.1):
    n = len(stack); nrow = int(math.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * cell, nrow * cell))
    axes = np.atleast_1d(axes).ravel()
    for a in axes: a.axis("off")
    for i, m in enumerate(stack): axes[i].imshow(colorize(m))
    if title: fig.suptitle(title, fontsize=11)
    fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    return fig

# ---- distribution ----
def class_shares(stack):
    h = np.bincount(stack.ravel(), minlength=cfg.N_CODES).astype(np.float64)
    return h / max(h.sum(), 1.0)

def kl_div(p, q, eps=1e-8):
    p = np.clip(p, eps, None); p = p / p.sum()
    q = np.clip(q, eps, None); q = q / q.sum()
    return float((p * np.log(p / q)).sum())

# ---- landscape ----
LS_KEYS = ["n_patches", "mean_area", "largest_share", "edge_density", "share"]

def landscape_one(a):
    out, area = {}, a.size
    for c in range(cfg.N_CODES):
        m = a == c; npx = int(m.sum())
        if npx == 0:
            out[c] = dict(n_patches=0.0, mean_area=0.0, largest_share=0.0,
                          edge_density=0.0, share=0.0); continue
        lab, n = ndi.label(m, structure=STRUCT8)
        ar = np.bincount(lab.ravel())[1:]
        eh = int(((a[:, 1:] == c) ^ (a[:, :-1] == c)).sum())      # vertical class boundaries
        ev = int(((a[1:, :] == c) ^ (a[:-1, :] == c)).sum())      # horizontal ones
        out[c] = dict(n_patches=float(n), mean_area=float(ar.mean()),
                      largest_share=float(ar.max() / npx),
                      edge_density=1000.0 * (eh + ev) / area, share=npx / area)
    p = np.bincount(a.ravel(), minlength=cfg.N_CODES) / area
    p = p[p > 0]
    return out, float(-(p * np.log(p)).sum())                     # Shannon diversity (SHDI)

def landscape_table(stack, name):
    acc = {c: {k: [] for k in LS_KEYS} for c in range(cfg.N_CODES)}; shdi = []
    for a in stack:
        per, s = landscape_one(a); shdi.append(s)
        for c, d in per.items():
            for k in LS_KEYS: acc[c][k].append(d[k])
    rows = []
    for c in range(cfg.N_CODES):
        r = {"code": c, "cls": CODE_NAMES[c], "set": name}
        for k in LS_KEYS:
            v = np.asarray(acc[c][k], dtype=np.float64)
            r[f"{k}_mean"] = float(v.mean()); r[f"{k}_sd"] = float(v.std())
        rows.append(r)
    return pd.DataFrame(rows), np.asarray(shdi)

# ---- connectivity of the thin classes ----
THIN = {1: "Road", 2: "River"}

def connectivity_stats(stack, c):
    rows = []
    for a in stack:
        m = a == c; npx = int(m.sum())
        if npx == 0:
            rows.append(dict(present=0, n_comp=0, frag=np.nan, mean_area=np.nan,
                             largest_share=np.nan, share=0.0)); continue
        lab, n = ndi.label(m, structure=STRUCT8)
        ar = np.bincount(lab.ravel())[1:]
        rows.append(dict(present=1, n_comp=int(n), frag=1000.0 * n / npx,
                         mean_area=float(ar.mean()), largest_share=float(ar.max() / npx),
                         share=npx / a.size))
    return pd.DataFrame(rows)

# ---- memorization ----
def nn_distance(query, ref, size=32, chunk=64):
    # min per-pixel disagreement of each query mask against the whole reference set
    def flat(s):
        return np.stack([cv2.resize(a, (size, size), interpolation=cv2.INTER_NEAREST).ravel()
                         for a in s])
    Q, R = flat(query), flat(ref)
    d = np.empty(len(Q), dtype=np.float64)
    for i in range(0, len(Q), chunk):
        d[i:i + chunk] = (Q[i:i + chunk, None, :] != R[None, :, :]).mean(-1).min(1)
    return d

# ---- post-processing: despeckle, then distribution-matched subset selection ----
def smooth_mask(a, modal_k=None, close_k=None, min_px_thin=None, min_px_bulk=None,
                thin=(1, 2)):
    """Polygon-ise a generated mask. Replaces the delete-only despeckle of the first run.

    1. Lift Road/River out and refill their footprint from the nearest bulk label, so the
       modal filter cannot erase a 2 px-wide line (a thin class is always the minority in a
       3x3 window -- running the filter on the full mask deletes every road).
    2. Modal (majority) filter the bulk layer: smooths ragged boundaries, kills isolated
       pixels. This is what moves n_patches and SHDI back toward real.
    3. Per thin class: binary closing to BRIDGE GAPS, then drop components below
       min_px_thin. Closing is what frag_ratio needs; deletion alone cannot help it.
    4. Paint Road, then River, back on top -- River last, so a river crossing a road stays
       a river, matching how the source polygons were digitized.
    """
    modal_k    = cfg.PP_MODAL if modal_k is None else modal_k
    close_k    = cfg.PP_CLOSE if close_k is None else close_k
    min_px_thin= cfg.PP_THIN  if min_px_thin is None else min_px_thin
    min_px_bulk= cfg.PP_BULK  if min_px_bulk is None else min_px_bulk

    a = a.astype(np.uint8)
    tm = {c: (a == c) for c in thin}
    any_thin = np.zeros_like(a, bool)
    for m in tm.values(): any_thin |= m

    bulk = a.copy()
    if any_thin.any() and (~any_thin).any():
        _, ix = ndi.distance_transform_edt(any_thin, return_distances=True, return_indices=True)
        bulk[any_thin] = bulk[tuple(i[any_thin] for i in ix)]

    if modal_k and modal_k > 1:
        k = np.ones((modal_k, modal_k), np.float32) / (modal_k * modal_k)
        votes = np.stack([ndi.convolve((bulk == c).astype(np.float32), k, mode="nearest")
                          for c in range(cfg.N_CODES)])
        # tie-break toward the incumbent label so the filter cannot drift the class marginal
        rr, cc = np.arange(bulk.shape[0])[:, None], np.arange(bulk.shape[1])[None]
        votes[bulk, rr, cc] += 1e-3
        bulk = votes.argmax(0).astype(np.uint8)

    if min_px_bulk:
        for c in range(cfg.N_CODES):
            if c in thin: continue
            m = bulk == c
            if not m.any(): continue
            lab, _ = ndi.label(m, structure=STRUCT8)
            ar = np.bincount(lab.ravel()); ar[0] = 0
            small = np.where((ar > 0) & (ar < min_px_bulk))[0]
            if len(small):
                drop = np.isin(lab, small)
                if (~drop).any():
                    _, ix = ndi.distance_transform_edt(drop, return_distances=True,
                                                       return_indices=True)
                    bulk[drop] = bulk[tuple(i[drop] for i in ix)]

    out = bulk
    for c in thin:                                   # Road first, River last
        m = tm[c]
        if not m.any(): continue
        if close_k and close_k > 1:
            m = ndi.binary_closing(m, structure=np.ones((close_k, close_k), bool))
        if min_px_thin:
            lab, _ = ndi.label(m, structure=STRUCT8)
            ar = np.bincount(lab.ravel()); ar[0] = 0
            m = np.isin(lab, np.where(ar >= min_px_thin)[0])
        out = np.where(m, c, out).astype(np.uint8)
    return out

def despeckle(a, **kw):                              # name kept: section 19 still calls this
    return smooth_mask(a, **kw)

def frag_of(stack, c):
    v = [1000.0 * ndi.label(a == c, structure=STRUCT8)[1] / (a == c).sum()
         for a in stack if (a == c).any()]
    return float(np.mean(v)) if v else float("nan")

def pp_report(stack, ref):
    p_r, p_g = class_shares(ref), class_shares(stack)
    d = {"kl": kl_div(p_r, p_g)}
    for c, nm in [(1, "Road"), (2, "River")]:
        fr, fg = frag_of(ref, c), frag_of(stack, c)
        d[f"frag_{nm}"]  = fg / fr if fr else float("nan")
        d[f"share_{nm}"] = p_g[c] / p_r[c] if p_r[c] else float("nan")
    d["npatch"] = float(np.mean([
        sum(ndi.label(a == c, structure=STRUCT8)[1] for c in range(cfg.N_CODES) if (a == c).any())
        for a in stack]))
    sh = [np.bincount(a.ravel(), minlength=cfg.N_CODES) / a.size for a in stack]
    d["shdi"] = float(np.mean([-(q[q > 0] * np.log(q[q > 0])).sum() for q in sh]))
    return d

def pp_sweep(pool, ref, n=None):
    """Try the four knobs on a subsample, print the table, return the winning config.

    Admissible = both thin classes inside the section-19 gates and the class marginal not
    made worse. Among those, pick the one whose overall patch structure is closest to real:
    frag_ratio is the gate, but n_patches / SHDI is what makes Stage 2's job honest.
    """
    n = cfg.PP_SWEEP_N if n is None else n
    sub = pool[np.random.RandomState(0).choice(len(pool), min(n, len(pool)), replace=False)]
    base, tgt = pp_report(sub, ref), pp_report(ref, ref)
    hdr = f"{'config':<32}{'KL':>8}{'fragRoad':>10}{'fragRiv':>9}{'shRoad':>8}{'shRiv':>8}{'npatch':>8}{'SHDI':>7}"
    print("\n--- post-processing sweep ---"); print(hdr)
    print(f"{'REAL (target)':<32}{0.0:8.5f}{1.0:10.2f}{1.0:9.2f}{1.0:8.2f}{1.0:8.2f}"
          f"{tgt['npatch']:8.1f}{tgt['shdi']:7.3f}")
    print(f"{'raw pool (no post-processing)':<32}{base['kl']:8.5f}{base['frag_Road']:10.2f}"
          f"{base['frag_River']:9.2f}{base['share_Road']:8.2f}{base['share_River']:8.2f}"
          f"{base['npatch']:8.1f}{base['shdi']:7.3f}")
    print("-" * len(hdr))
    res = []
    for mk in (0, 3, 5):
        for ck in (0, 3):
            for mp in (12, 30, 60):
                for mb in (0, 40):
                    st = np.stack([smooth_mask(a, mk, ck, mp, mb) for a in sub])
                    r = pp_report(st, ref); r["cfg"] = (mk, ck, mp, mb); res.append(r)
                    print(f"{f'modal{mk} close{ck} thin{mp} bulk{mb}':<32}{r['kl']:8.5f}"
                          f"{r['frag_Road']:10.2f}{r['frag_River']:9.2f}{r['share_Road']:8.2f}"
                          f"{r['share_River']:8.2f}{r['npatch']:8.1f}{r['shdi']:7.3f}", flush=True)
    ok = [r for r in res
          if r["frag_Road"] <= 3.0 and r["frag_River"] <= 3.0
          and 0.25 <= r["share_Road"] <= 2.5 and 0.25 <= r["share_River"] <= 2.5
          and r["kl"] <= 2 * base["kl"]]
    print("-" * len(hdr))
    if ok:
        b = min(ok, key=lambda r: abs(np.log(r["npatch"] / tgt["npatch"]))
                                  + abs(np.log(r["shdi"] / tgt["shdi"])))
        print(f"{len(ok)}/{len(res)} configs pass both thin-class gates. "
              f"CHOSEN: modal{b['cfg'][0]} close{b['cfg'][1]} thin{b['cfg'][2]} bulk{b['cfg'][3]}"
              f"  (River frag {b['frag_River']:.2f}, was {base['frag_River']:.2f})")
        return b["cfg"]
    print("NO config passes both gates. River fragmentation is NOT a post-processing problem:")
    print("  at 128 px a river is ~3 px wide and the model cannot hold a continuous channel.")
    print("  This is a real negative result -- report the FAIL and put the resolution limit in")
    print("  the discussion. Falling back to the CFG defaults for the hand-off set.")
    return (cfg.PP_MODAL, cfg.PP_CLOSE, cfg.PP_THIN, cfg.PP_BULK)

def greedy_match(stack, target, k):
    # Pick k of the pool so the SELECTED set's class marginal is closest to the real one in KL.
    # Minimising L1 instead would happily drop River entirely -- KL punishes that, L1 does not.
    sh = np.stack([np.bincount(a.ravel(), minlength=cfg.N_CODES) / a.size for a in stack])
    t = np.clip(target, 1e-8, None); t = t / t.sum()
    avail, chosen, cur = list(range(len(stack))), [], np.zeros(cfg.N_CODES)
    for step in range(min(k, len(stack))):
        cand = (cur * step + sh[avail]) / (step + 1)
        c = np.clip(cand, 1e-8, None); c = c / c.sum(1, keepdims=True)
        j = int((t[None] * np.log(t[None] / c)).sum(1).argmin())
        cur = cand[j]; chosen.append(avail.pop(j))
    return chosen, cur

_f = mask_grid(TRAIN_128[:8], ncol=4, title=f"real fold-{cfg.FOLD} train masks (128, class indices)")
plt.show()
print("metrics module ready | thin classes:", THIN)

## 14. Efficiency metrics

Params, GFLOPs of one UNet evaluation at 128x128, GFLOPs per *sample* (that is the UNet cost times
`DDIM_STEPS` — the number that actually matters for a diffusion model), sampling throughput in
masks/minute, and peak training VRAM.

Two wrinkles worth naming:
- `ptflops` needs a module whose `forward` takes a single tensor, so `UNet2DModel` is wrapped with a
  zero timestep. Timestep embedding is a negligible share of the cost, so this does not bias the
  figure.
- `ptflops` hooks `nn.Module`s and therefore **misses the functional attention matmuls**. The two
  attention levels here are at 32x32 and 16x16, so the undercount is small, but say so in the
  caption. The throughput measurement below has no such caveat and is the honest cost figure to
  report for generation.

In [ ]:
class _FlopsWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        t = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
        return self.m(x, t).sample

def efficiency_report(m):
    from ptflops import get_model_complexity_info
    m.eval()
    w = _FlopsWrapper(m)
    with torch.cuda.amp.autocast(enabled=False):
        macs, params = get_model_complexity_info(
            w, (cfg.N_CH, cfg.GEN_SIZE, cfg.GEN_SIZE),
            as_strings=False, print_per_layer_stat=False, verbose=False)
    x = torch.randn(cfg.GEN_BATCH, cfg.N_CH, cfg.GEN_SIZE, cfg.GEN_SIZE, device="cuda")
    t = torch.zeros(cfg.GEN_BATCH, dtype=torch.long, device="cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(5): m(x, t)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(20): m(x, t)
        torch.cuda.synchronize()
    unet_s = (time.time() - t0) / 20                      # one UNet eval on GEN_BATCH masks
    per_mask_s = unet_s * cfg.DDIM_STEPS / cfg.GEN_BATCH
    return {"params_M": params / 1e6,
            "GFLOPs_unet_128": macs * 2 / 1e9,            # ptflops returns MACs; FLOPs = 2 x MACs
            "GFLOPs_per_sample": macs * 2 / 1e9 * cfg.DDIM_STEPS,
            "masks_per_min": 60.0 / per_mask_s,
            "est_minutes_for_N_GENERATE": cfg.N_GENERATE * per_mask_s / 60.0}

_eff = efficiency_report(model)
print(json.dumps(_eff, indent=2))
print(f"\n-> generating {cfg.N_GENERATE} masks will take about "
      f"{_eff['est_minutes_for_N_GENERATE']:.1f} min at DDIM {cfg.DDIM_STEPS} steps")
print(f"-> selection sampling costs about "
      f"{cfg.SEL_N_SAMPLE * _eff['est_minutes_for_N_GENERATE'] / cfg.N_GENERATE:.2f} min "
      f"every {cfg.SAMPLE_EVERY} epochs")
_pool_min = (_eff["est_minutes_for_N_GENERATE"] * cfg.N_POOL / cfg.N_GENERATE
             * cfg.FINAL_DDIM_STEPS / cfg.DDIM_STEPS)
print(f"-> section 19 will sample a pool of {cfg.N_POOL} at {cfg.FINAL_DDIM_STEPS} steps: "
      f"about {_pool_min:.0f} min")
if _pool_min > 120:
    print("   WARNING: that is over 2 h on top of training. Drop FINAL_DDIM_STEPS to 100 or "
          "N_POOL to 2500 before you run out of session time.")


## 15. Hugging Face persistence and resume

Same two-artifact layout as the segmentation notebooks, in this run's own repo
(`geonusaf-stage1-maskddpm-block-fold0`):

- `best.pt` — EMA weights (selected on sampled class-share KL) + the full `UNet2DModel` config
  + the run `cfg` + metrics. This is what
  section 22's standalone sampler downloads, and what Stage 2 would pull if it ever needs to
  regenerate masks itself.
- `last.pt` — full state incl. optimizer/scaler/scheduler/EMA, pushed every `PUSH_LAST_EVERY`
  epochs. This is what a new session resumes from.

`last.pt` here is ~500 MB (model + EMA + Adam moments for a 45 M-param UNet), which is why
`PUSH_LAST_EVERY` is 25 rather than the segmentation notebooks' 10 — pushing every 10 epochs would
spend more of the session uploading than training.

**The architecture signature is checked on load.** Silently loading weights trained under a
different channel layout is the sort of failure that produces plausible-looking noise for six hours,
so `load_state` raises rather than warns.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local_path, name, msg):
    try:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                        repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)
    except Exception as e:
        print("  HF push failed (continuing):", type(e).__name__, e)

def hf_try_download(name):
    try: return hf_hub_download(HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

def check_arch(st):
    got = st.get("arch")
    if got != ARCH_SIG:
        raise RuntimeError("architecture signature mismatch, refusing to load\n"
                           f"  checkpoint: {got}\n  this run  : {ARCH_SIG}\n"
                           "  (a run-A binary-code checkpoint will land here -- that is correct, "
                           "the two runs are separate arms and must not share weights)")

def push_model_card(final, epoch):
    L = ["---",
         "tags: [diffusion, ddpm, label-generation, remote-sensing, geonusaf]",
         "library_name: diffusers", "---", "",
         f"# GeoNUSAF Stage 1 - unconditional land-use mask DDPM ({cfg.SPLIT_MODE} fold {cfg.FOLD})",
         "",
         "Generates novel Kathmandu Valley land-use layouts as single-channel masks. Trained on the",
         f"fold-{cfg.FOLD} TRAIN tiles only; validation tiles never enter a gradient.", "",
         "**Arm B of the label-representation ablation: one-hot channels.** Arm A used the SatSynth",
         "binary code and is in a separate repo.", "",
         "| field | value |", "|---|---|",
         f"| architecture | diffusers UNet2DModel, {n_par/1e6:.1f} M params |",
         f"| label coding | **{cfg.LABEL_CODING}**, {cfg.N_CH} channels, {cfg.N_CODES} codes |",
         f"| diffusion | DDPM T={cfg.TIMESTEPS}, linear beta, **{cfg.PREDICTION_TYPE}**, "
         f"zero-terminal-SNR {cfg.ZERO_TERMINAL_SNR}, offset noise {cfg.OFFSET_NOISE} |",
         f"| sampling | DDIM {cfg.DDIM_STEPS} steps, eta {cfg.DDIM_ETA} |",
         f"| resolution | trained {cfg.GEN_SIZE}, exported {cfg.OUT_SIZE} (INTER_NEAREST) |",
         f"| split | {cfg.SPLIT_MODE} fold {cfg.FOLD}, train_sha1 {split_sha1[:12]} |",
         f"| model selection | {cfg.SELECT_ON} (sampled class-share KL), not the denoising loss |",
         f"| River oversampling | {cfg.RIVER_OVERSAMPLE} |",
         f"| seed | {cfg.SEED} |",
         f"| epochs | {epoch} |",
         f"| KL(real\\|\\|gen) | {final.get('KL_real_gen', float('nan')):.5f} |",
         f"| ambiguity rate | {final.get('ambiguity_rate', float('nan')):.5f} |",
         f"| recon acc @ t={cfg.RECON_T} | {final.get('recon', {}).get('pixel_acc', float('nan')):.4f} |",
         "",
         "## Output contract (Stage 2 input)",
         f"`genmask{{i}}.png`, single channel, {cfg.OUT_SIZE}x{cfg.OUT_SIZE}, "
         "values `{0..5}` + `255` for ignore.", "",
         "## Class shares (generated vs real fold train)",
         "| class | real % | gen % | ratio |", "|---|---|---|---|"]
    for i, nm in enumerate(CODE_NAMES):
        r = final.get("class_share_real", {}).get(nm, float("nan"))
        gq = final.get("class_share_gen", {}).get(nm, float("nan"))
        L.append(f"| {nm} | {100*r:.2f} | {100*gq:.2f} | {(gq/r if r else float('nan')):.2f} |")
    L += ["", "## Thin-class connectivity (generated vs real)",
          "| class | frag gen | frag real | ratio | verdict |", "|---|---|---|---|---|"]
    for r in final.get("connectivity", []):
        L.append(f"| {r['cls']} | {r['frag_gen']:.3f} | {r['frag_real']:.3f} | "
                 f"{r['frag_ratio']:.2f} | {r['verdict']} |")
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write("\n".join(L))
    hf_push(p, "README.md", f"card epoch {epoch}")

print("HF ready:", HF_REPO)


## 16. Weights & Biases

Matches the logging on the segmentation baselines. `resume="allow"` with a deterministic run id
means a resumed session continues the same run rather than starting a second one.

Two series here that the segmentation notebooks do not have:
- **`ambiguity_rate`** (one-hot has no illegal state; this is its diagnostic replacement)
- **`sel_kl`, `sel_share_River`, `sel_share_Road`** — the selection series
- (run A's **`illegal_code_rate`** — the share of sampled pixels landing on the unused code 7 before
  nearest-code projection. It is the cheapest available proxy for "how far are the samples from the
  code corners", and it should fall steadily.
- **`sample grids`** every `SAMPLE_EVERY` epochs. For a generative model the grid *is* the training
  curve; the loss is nearly flat and uninformative after the first few hundred steps, which is
  normal for DDPM and not a sign that training has stalled.

In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
               resume="allow", config=asdict(cfg))
    wandb.config.update({"arch_sig": ARCH_SIG, "n_params": n_par,
                         "train_tiles": len(tr_ids), "train_masks": len(train_ds),
                         "split_sha1": split_sha1}, allow_val_change=True)
print("wandb:", WANDB)

## 17. Train step, monitors, sampler, and the two diagnostics run A did not have

### Training
One step is: draw `t ~ U{0..999}`, draw `eps ~ N(0,I)`, form `x_t`, predict **v**, take the MSE.
The loss is computed **in fp32 outside autocast** for the same reason the segmentation notebooks do
it — fp16 MSE on a near-zero residual loses most of its significant digits.

### `validate()` — a curve, not a selector
Run A drew a random `t` per batch under a fixed seed and selected `best.pt` on the result. That
number is dominated by the high-`t` regime, where every model looks identical, and it chose epoch
536 for a model whose samples were unusable. Here the timesteps are **stratified** across
`[0, T)` so the estimate is an honest Monte-Carlo average, and a second series `val_loss_lowt`
restricted to `t < VAL_LOWT` tracks the detail regime that actually governs thin structures.
Neither drives selection.

### `sample_quality()` — what does drive selection
Every `SAMPLE_EVERY` epochs the EMA weights sample `SEL_N_SAMPLE` masks and the run records:
class-share **KL against the real fold-1 train marginal**, per-class share for River and Road, the
fraction of masks containing any River, the ambiguity rate, and Shannon diversity. `best.pt` is
saved on KL. This is the number that would have caught run A at epoch 50 instead of epoch 699.

### `recon_check()` — the five-minute triage
Noise real masks to `t = RECON_T`, denoise back with DDIM, decode, report pixel accuracy. It
separates two failure modes that look identical in a sample grid:
- **high recon accuracy, bad samples** → the denoiser is fine, the prior/schedule is wrong,
- **low recon accuracy** → the model is simply undertrained; give it more steps.

Run this before spending another 20 hours on any hypothesis.


In [ ]:
def train_one_epoch(epoch):
    model.train(); train_ds.epoch = epoch
    tot, n, gn_acc = 0.0, 0, []
    optimizer.zero_grad(set_to_none=True)
    for it, x0 in enumerate(train_ld):
        x0 = x0.cuda(non_blocking=True)
        t = torch.randint(0, cfg.TIMESTEPS, (x0.size(0),), device="cuda")
        noise = torch.randn_like(x0)
        if cfg.OFFSET_NOISE:   # off by default: on one-hot channels the per-channel mean IS the
                               # class marginal, and sampling from plain randn would not match
            noise = noise + cfg.OFFSET_NOISE * torch.randn(
                x0.size(0), x0.size(1), 1, 1, device="cuda")
        xt  = sched.q_sample(x0, t, noise)
        tgt = sched.target(x0, t, noise)                          # v, or eps if PREDICTION_TYPE says so
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            pred = net(xt, t)                                     # net == model, or DP over 2 GPUs
        loss = F.mse_loss(pred.float(), tgt)                      # fp32, outside autocast
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()

        if (it + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)                            # unscale BEFORE clipping
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
            gn_acc.append(float(gn))
            prev_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            if ema is not None and scaler.get_scale() >= prev_scale:  # skip EMA on skipped steps
                ema.update(model, scheduler.last_epoch)

        tot += float(loss) * x0.size(0); n += x0.size(0)
    return tot / max(n, 1), float(np.mean(gn_acc)), float(np.max(gn_acc))


@torch.no_grad()
def validate(m=None):
    # Stratified over t, not a random draw: an honest average, plus the low-t detail regime.
    m = m if m is not None else eval_model
    m.eval(); tot, n, tot_lo, n_lo = 0.0, 0, 0.0, 0
    for bi, x0 in enumerate(val_ld):
        x0 = x0.cuda(non_blocking=True); b = x0.size(0)
        gg = torch.Generator(); gg.manual_seed(cfg.VAL_NOISE_SEED + bi)
        noise = torch.randn(x0.shape, generator=gg).cuda()
        # evenly spaced timesteps, offset by batch index so the union covers [0, T) densely
        t = ((torch.arange(b, device="cuda") + bi * 0.37) * cfg.TIMESTEPS / b).long() % cfg.TIMESTEPS
        xt, tgt = sched.q_sample(x0, t, noise), sched.target(x0, t, noise)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            pred = m(xt, t).sample
        per = ((pred.float() - tgt) ** 2).mean(dim=(1, 2, 3))
        tot += float(per.sum()); n += b
        lo = t < cfg.VAL_LOWT
        if lo.any(): tot_lo += float(per[lo].sum()); n_lo += int(lo.sum())
    return tot / max(n, 1), (tot_lo / n_lo if n_lo else float("nan"))


@torch.no_grad()
def sample_masks(m, n, steps=None, seed=0, batch=None, progress=False):
    # returns (n, GEN_SIZE, GEN_SIZE) uint8 of CODES (ignore = 6) and the ambiguity rate
    m.eval()
    steps = steps or cfg.DDIM_STEPS
    batch = batch or cfg.GEN_BATCH
    out, rates = [], []
    for s in range(0, n, batch):
        b = min(batch, n - s)
        gg = torch.Generator(); gg.manual_seed(cfg.SEED + seed + s)
        x = torch.randn(b, cfg.N_CH, cfg.GEN_SIZE, cfg.GEN_SIZE, generator=gg).cuda()
        sc = build_ddim(); sc.set_timesteps(steps)          # fresh scheduler per batch
        for t in sc.timesteps:
            with torch.cuda.amp.autocast(enabled=cfg.AMP):
                pred = m(x, t).sample
            x = sc.step(pred.float(), t, x, eta=cfg.DDIM_ETA).prev_sample
        idx, r = decode_codes(x)
        out.append(idx.to(torch.uint8).cpu().numpy()); rates.append(r)
        if progress: print(f"  sampled {s+b}/{n}", flush=True)
    return np.concatenate(out), float(np.mean(rates))


P_REAL_TRAIN = class_shares(TRAIN_128)         # the marginal the generator is asked to reproduce

def sample_quality(stack, amb_rate):
    # The selection metric. KL is the headline; the rest are the series that explain a bad KL.
    p = class_shares(stack)
    shdi = []
    for a in stack:
        q = np.bincount(a.ravel(), minlength=cfg.N_CODES) / a.size
        q = q[q > 0]; shdi.append(float(-(q * np.log(q)).sum()))
    d = {"kl": kl_div(P_REAL_TRAIN, p), "amb_rate": float(amb_rate),
         "shdi": float(np.mean(shdi)),
         "frac_with_River": float(np.mean([(a == 2).any() for a in stack])),
         "frac_with_Road": float(np.mean([(a == 1).any() for a in stack]))}
    for i in range(cfg.N_CODES):
        d[f"share_{CODE_NAMES[i]}"] = float(p[i])
    return d


@torch.no_grad()
def recon_check(m, t_start=None, n=64, steps=None, seed=3):
    # Noise real masks to t_start, denoise back, decode, report pixel accuracy against the original.
    # High accuracy + bad samples => the denoiser is fine and the prior/schedule is at fault.
    # Low accuracy => undertrained. Cheap, and it settles the argument in five minutes.
    t_start = cfg.RECON_T if t_start is None else t_start
    steps = steps or cfg.DDIM_STEPS
    m.eval()
    rs = np.random.RandomState(seed)
    sel = rs.choice(len(TRAIN_128), min(n, len(TRAIN_128)), replace=False)
    accs, ious = [], []
    for s in range(0, len(sel), cfg.GEN_BATCH):
        idx0 = sel[s:s + cfg.GEN_BATCH]
        lab0 = TRAIN_128[idx0]
        x0 = torch.from_numpy(np.stack([encode_index(a) for a in lab0])).cuda()
        gg = torch.Generator(); gg.manual_seed(cfg.SEED + seed + s)
        noise = torch.randn(x0.shape, generator=gg).cuda()
        tt = torch.full((x0.size(0),), t_start, device="cuda", dtype=torch.long)
        x = sched.q_sample(x0, tt, noise)
        sc = build_ddim(); sc.set_timesteps(steps)
        for t in sc.timesteps:
            if int(t) > t_start: continue
            with torch.cuda.amp.autocast(enabled=cfg.AMP):
                pred = m(x, t).sample
            x = sc.step(pred.float(), t, x, eta=0.0).prev_sample
        rec, _ = decode_codes(x)
        rec = rec.cpu().numpy()
        accs.append((rec == lab0).mean())
        for c in range(cfg.N_CODES):
            u = ((rec == c) | (lab0 == c)).sum()
            if u: ious.append((c, ((rec == c) & (lab0 == c)).sum() / u))
    per_c = {CODE_NAMES[c]: float(np.mean([v for k, v in ious if k == c]))
             for c in range(cfg.N_CODES) if any(k == c for k, _ in ious)}
    return {"recon_t": t_start, "pixel_acc": float(np.mean(accs)), "per_class_iou": per_c}


_t0 = time.time()
_s, _r = sample_masks(eval_model, 4)
print(f"smoke sample: {_s.shape} in {time.time()-_t0:.1f}s | ambiguity rate {_r:.4%} "
      "(untrained -> expect noise)")
print("selection metric on the untrained model:",
      {k: round(v, 4) for k, v in sample_quality(_s, _r).items() if k in ("kl", "shdi", "amb_rate")})


## 18. Main loop with automatic resume

On start this pulls `last.pt` from the repo if it exists and continues from that epoch, with the
architecture signature checked first. `ARCH_SIG` now carries the label coding, the prediction type
and the offset-noise setting, so a run-A checkpoint is refused here rather than silently loaded
into a 7-channel UNet.

At 25 steps per epoch, `MAX_EPOCHS=1600` is 40 k steps. Session 1 reached epoch ~1100 in 10.5 h;
the remaining ~500 epochs are about 4.5 h, one more session. The horizon was cut from 2400 to 1600
so the cosine actually **finishes** — a truncated 2400-epoch schedule leaves the LR at 1.2e-4 and
never anneals, and the annealed tail is where sample quality consolidates.

### What changed against run A
- **Selection is on `sel_kl`**, the class-share KL of a freshly sampled set against the real fold-1
  train marginal, evaluated every `SAMPLE_EVERY` epochs after `SEL_WARMUP_EPOCHS`. `best_val` in
  the checkpoint dict now holds that KL, not a denoising loss.
- **Early stopping is off.** It was triggering on a metric that does not measure sample quality;
  with a cosine schedule the last epochs are also the most useful ones.
- `val_loss` and `val_loss_lowt` are still logged, because a rising low-t curve alongside a flat
  KL is the honest signature of overfitting on 268 base tiles — and that is a claim run A could not
  actually support.
- **A River admissibility gate on selection.** Road and River are only ~8% of the KL, so a
  checkpoint that has stopped drawing rivers can still post the best KL. The gate does not change
  the metric — it makes a checkpoint outside `[RIVER_GATE_LO, RIVER_GATE_HI]` x the real River
  share ineligible to be crowned. KL stays directly comparable to run A.
- **`RESET_BEST_ON_RESUME`**, a one-shot flag for the resume where `SEL_N_SAMPLE` changes.


In [ ]:
ck = hf_try_download("last.pt")
start_epoch, best_val, best_epoch, history = 0, float("inf"), -1, []
if ck:
    st = torch.load(ck, map_location="cuda")
    check_arch(st)
    model.load_state_dict(st["model_state"]); optimizer.load_state_dict(st["optim_state"])
    scheduler.load_state_dict(st["sched_state"]); scaler.load_state_dict(st["scaler_state"])
    if ema is not None and st.get("ema_state") is not None:
        ema.ema.load_state_dict(st["ema_state"])
    start_epoch = st["epoch"] + 1; best_val = st["best_val"]; best_epoch = st["best_epoch"]
    history = st["history"]
    if cfg.RESET_BEST_ON_RESUME:
        # The inherited best_val was measured with a different SEL_N_SAMPLE. Re-open the contest
        # rather than let an old lucky draw block every future checkpoint. best.pt on the hub is
        # left in place, so if nothing beats it this session you still have a usable checkpoint.
        print(f"  RESET_BEST_ON_RESUME: dropping inherited best {best_val:.5f} @ e{best_epoch}")
        best_val, best_epoch = float("inf"), -1
    # best.pt lives on the hub from the session that set it; without this, a session that never
    # beats the inherited best has no local best.pt to push or to evaluate in section 19.
    _b = hf_try_download("best.pt")
    if _b: shutil.copy(_b, f"{cfg.OUT_DIR}/best.pt")
    print(f"RESUMED from epoch {st['epoch']} (best {cfg.SELECT_ON} {best_val:.5f} @ e{best_epoch}) | "
          f"best.pt restored: {bool(_b)}")
else:
    print("fresh start")

torch.cuda.reset_peak_memory_stats()
t_start = time.time()
epoch = start_epoch - 1

for epoch in range(start_epoch, cfg.MAX_EPOCHS):
    te = time.time()
    tr_loss, gn_mean, gn_max = train_one_epoch(epoch)
    va_loss, va_lowt = validate() if cfg.VAL_MONITOR else (float("nan"), float("nan"))
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, "val_loss_lowt": va_lowt,
           "overfit_gap": va_loss - tr_loss, "grad_norm_mean": gn_mean, "grad_norm_max": gn_max,
           "lr": scheduler.get_last_lr()[0], "peak_vram_gb": peak_gb,
           "epoch_sec": time.time() - te, "hours": (time.time() - t_start) / 3600}

    # ---- sampling-based monitor and model selection ----
    sel = None
    if (epoch + 1) % cfg.SAMPLE_EVERY == 0:
        s, r = sample_masks(eval_model, cfg.SEL_N_SAMPLE, steps=cfg.SEL_DDIM_STEPS, seed=epoch)
        sel = sample_quality(s, r)
        row.update({f"sel_{k}": v for k, v in sel.items()})
        fig = mask_grid(s[:cfg.N_SAMPLE_GRID], ncol=4,
                        title=f"EMA samples - epoch {epoch} (KL {sel['kl']:.4f}, "
                              f"amb {sel['amb_rate']:.3%})")
        pg = f"{cfg.OUT_DIR}/samples_e{epoch}.png"
        plt.savefig(pg, dpi=110, bbox_inches="tight"); plt.show()
        if WANDB: wandb.log({"samples": wandb.Image(pg)}, step=epoch)
        print(f"  [e{epoch}] KL {sel['kl']:.5f} | amb {sel['amb_rate']:.4%} | "
              f"River {100*sel['share_River']:.2f}% (real {100*P_REAL_TRAIN[2]:.2f}%) | "
              f"Road {100*sel['share_Road']:.2f}% (real {100*P_REAL_TRAIN[1]:.2f}%) | "
              f"SHDI {sel['shdi']:.3f} | masks with River {sel['frac_with_River']:.2f}", flush=True)

    history.append(row)
    if WANDB: wandb.log(row, step=epoch)

    if epoch % 10 == 0 or epoch == cfg.MAX_EPOCHS - 1:
        print(f"[e{epoch}] train {tr_loss:.5f} | val {va_loss:.5f} (lowt {va_lowt:.5f}) | "
              f"gnorm mu{gn_mean:.2f}/max{gn_max:.2f} | lr {row['lr']:.2e} | "
              f"{row['epoch_sec']:.0f}s | vram {peak_gb:.1f}GB | {row['hours']:.2f}h", flush=True)

    # Selection. On "kl" this only fires at sampling epochs past the warmup; on "val_loss" it
    # reproduces run A's behaviour, which is kept only so the ablation can be run honestly.
    if cfg.SELECT_ON == "kl":
        # River admissibility gate -- see the CFG comment. A checkpoint outside the band is simply
        # not eligible; the score it would have posted is discarded, not penalised.
        _riv = (sel["share_River"] / P_REAL_TRAIN[2]) if sel is not None else float("nan")
        _ok = (sel is not None and epoch >= cfg.SEL_WARMUP_EPOCHS
               and cfg.RIVER_GATE_LO <= _riv <= cfg.RIVER_GATE_HI)
        if sel is not None and not _ok and epoch >= cfg.SEL_WARMUP_EPOCHS:
            print(f"  [e{epoch}] not eligible: River {_riv:.2f}x real, outside "
                  f"[{cfg.RIVER_GATE_LO}, {cfg.RIVER_GATE_HI}] (KL was {sel['kl']:.5f})", flush=True)
        score = sel["kl"] if _ok else float("nan")
    else:
        score = va_loss if cfg.VAL_MONITOR else tr_loss
    if score == score and score < best_val:                  # NaN-safe
        best_val, best_epoch = score, epoch
        p = f"{cfg.OUT_DIR}/best.pt"
        torch.save({"model_state": eval_model.state_dict(),   # EMA weights when EMA is on
                    "model_config": dict(model.config), "arch": ARCH_SIG,
                    "cfg": asdict(cfg), "epoch": epoch, "select_on": cfg.SELECT_ON,
                    "val_loss": va_loss, "sel_kl": (sel or {}).get("kl", float("nan")),
                    "best_score": best_val}, p)
        print(f"  -> new best {cfg.SELECT_ON} {best_val:.5f} @ e{epoch}", flush=True)

    last_epoch = (epoch == cfg.MAX_EPOCHS - 1)
    out_of_time = (time.time() - t_start) / 3600 > cfg.MAX_HOURS
    if (epoch % cfg.PUSH_LAST_EVERY == 0) or last_epoch or out_of_time:
        p = f"{cfg.OUT_DIR}/last.pt"
        torch.save({"model_state": model.state_dict(), "optim_state": optimizer.state_dict(),
                    "sched_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
                    "ema_state": (ema.ema.state_dict() if ema is not None else None),
                    "arch": ARCH_SIG, "epoch": epoch, "best_val": best_val,
                    "best_epoch": best_epoch, "history": history}, p)
        hf_push(p, "last.pt", f"state e{epoch}")
        if os.path.exists(f"{cfg.OUT_DIR}/best.pt"):
            hf_push(f"{cfg.OUT_DIR}/best.pt", "best.pt",
                    f"best {cfg.SELECT_ON} {best_val:.5f} @ e{best_epoch}")

    if out_of_time:
        print(f"TIME BUDGET reached at epoch {epoch} ({cfg.MAX_HOURS} h) -- state pushed, "
              "re-run the notebook in a new session to continue"); break

wall_h = (time.time() - t_start) / 3600
print(f"training done | best {cfg.SELECT_ON} {best_val:.5f} @ epoch {best_epoch} | "
      f"this session {wall_h:.2f} h")

if cfg.RECON_CHECK:
    rc = recon_check(eval_model)
    print(f"\nreconstruction check @ t={rc['recon_t']}: pixel accuracy {rc['pixel_acc']:.4f}")
    print("  per-class IoU:", {k: round(v, 3) for k, v in rc["per_class_iou"].items()})
    print("  high accuracy with bad samples -> prior/schedule problem, not capacity;\n"
          "  low accuracy -> undertrained, give it more steps before changing anything else.")


## 19. Final generation and evaluation of the best checkpoint

Reloads `best.pt` (EMA weights, selected on sampled class-share KL) into a fresh model, samples
`N_POOL` masks at `FINAL_DDIM_STEPS`, filters down to `N_GENERATE`, and runs the full realism suite
against `EVAL_N_REAL` real fold-train masks. Train, not val, is the right comparison set: fidelity
here means "does the generator reproduce what it was shown".

`illegal_code_rate` is reported as a literal `0.0` — one-hot argmax has no illegal state, and the
field is kept only so this run's CSV row lines up with run A's in the ablation table. The number
that carries the same diagnostic weight is `ambiguity_rate`.

The reconstruction check is repeated here on the selected checkpoint, because it is the single
line that tells a reader whether a disappointing sample grid is a capacity problem or a prior
problem.


In [ ]:
_bp = f"{cfg.OUT_DIR}/best.pt"
if not os.path.exists(_bp):                      # best was set in an earlier session
    _r = hf_try_download("best.pt")
    assert _r, "no best.pt locally or on the hub -- nothing to evaluate"
    shutil.copy(_r, _bp); print("restored best.pt from", HF_REPO)
_ck = torch.load(_bp, map_location="cuda")
check_arch(_ck)
best_model = build_model().cuda()
best_model.load_state_dict(_ck["model_state"])
best_model.eval()
print(f"loaded best.pt from epoch {_ck['epoch']} "
      f"(selected on {_ck.get('select_on', 'kl')} = {_ck.get('best_score', float('nan')):.5f}, "
      f"val {_ck.get('val_loss', float('nan')):.5f})")

RECON = recon_check(best_model) if cfg.RECON_CHECK else {}
if RECON:
    print(f"reconstruction @ t={RECON['recon_t']}: pixel accuracy {RECON['pixel_acc']:.4f} | "
          "per-class IoU:", {k: round(v, 3) for k, v in RECON["per_class_iou"].items()})

t0 = time.time()
GEN128, ambiguity = sample_masks(best_model, cfg.N_POOL if cfg.FILTER_MASKS else cfg.N_GENERATE,
                                 steps=cfg.FINAL_DDIM_STEPS, seed=777, progress=True)
gen_min = (time.time() - t0) / 60
np.save(f"{cfg.OUT_DIR}/gen128.npy", GEN128)
print(f"generated {len(GEN128)} masks in {gen_min:.1f} min at {cfg.FINAL_DDIM_STEPS} DDIM steps | "
      f"ambiguity rate {ambiguity:.4%}")
if ambiguity > 0.02:
    print("  WARNING: >2% of pixels have their top two channels nearly tied. The samples are not "
          "settling on the code corners -- expect speckle along boundaries.")

rng = np.random.RandomState(cfg.SEED)
sel = rng.choice(len(TRAIN_128), min(cfg.EVAL_N_REAL, len(TRAIN_128)), replace=False)
REAL128 = TRAIN_128[sel]

# ---- filter the pool down to the hand-off set, and report what the filter bought ----
RAW128 = GEN128
PP_CFG = (cfg.PP_MODAL, cfg.PP_CLOSE, cfg.PP_THIN, cfg.PP_BULK)
if cfg.FILTER_MASKS:
    before = dict(kl=kl_div(class_shares(REAL128), class_shares(RAW128)),
                  road=frag_of(RAW128, 1), river=frag_of(RAW128, 2))
    if cfg.PP_SWEEP:
        PP_CFG = pp_sweep(RAW128, REAL128)
    CLEAN128 = np.stack([smooth_mask(a, *PP_CFG) for a in RAW128])
    keep, _ = greedy_match(CLEAN128, class_shares(REAL128), cfg.N_GENERATE)
    GEN128 = CLEAN128[np.array(sorted(keep))]
    after = dict(kl=kl_div(class_shares(REAL128), class_shares(GEN128)),
                 road=frag_of(GEN128, 1), river=frag_of(GEN128, 2))
    print(f"\nfilter: pool {len(RAW128)} -> {len(GEN128)} exported | "
          f"post-processing modal{PP_CFG[0]} close{PP_CFG[1]} thin{PP_CFG[2]} bulk{PP_CFG[3]}")
    print(f"  KL(real||gen)  {before['kl']:.4f} -> {after['kl']:.4f}")
    print(f"  Road frag      {before['road']:.1f} -> {after['road']:.1f}")
    print(f"  River frag     {before['river']:.1f} -> {after['river']:.1f}")
    print("  the filter corrects marginals post hoc; report the UNFILTERED KL as the model's own "
          "score, and the filtered one as the hand-off set's")
    np.save(f"{cfg.OUT_DIR}/gen128_unfiltered.npy", RAW128)
    np.save(f"{cfg.OUT_DIR}/gen128_smoothed.npy", CLEAN128)

# ---- 1. class proportions + KL ----
p_real, p_gen = class_shares(REAL128), class_shares(GEN128)
p_raw = class_shares(RAW128)
m_mix = 0.5 * (p_real + p_gen)
KL_RG, KL_GR = kl_div(p_real, p_gen), kl_div(p_gen, p_real)
KL_RAW = kl_div(p_real, p_raw)
JS = 0.5 * kl_div(p_real, m_mix) + 0.5 * kl_div(p_gen, m_mix)
prop = pd.DataFrame({"cls": CODE_NAMES, "real_pct": 100 * p_real,
                     "gen_raw_pct": 100 * p_raw, "gen_pct": 100 * p_gen})
prop["abs_diff"] = (prop.gen_pct - prop.real_pct).abs()
prop["ratio_raw"] = prop.gen_raw_pct / prop.real_pct.replace(0, np.nan)
prop["ratio"] = prop.gen_pct / prop.real_pct.replace(0, np.nan)
print("\n--- class proportions ---")
print(prop.round(4).to_string(index=False))
print(f"KL(real||gen) filtered {KL_RG:.5f} | unfiltered {KL_RAW:.5f} | "
      f"KL(gen||real) {KL_GR:.5f} | JS {JS:.5f}")
print("  run A, for the ablation table: River ratio 7.7, Forest ratio 0.54, illegal rate 0.00479")

# ---- 2. landscape metrics ----
LS_REAL, SH_REAL = landscape_table(REAL128, "real")
LS_GEN,  SH_GEN  = landscape_table(GEN128, "gen")
lsc = LS_REAL[["cls"]].copy()
for k in ["n_patches", "mean_area", "edge_density", "largest_share"]:
    lsc[f"{k}|real"] = LS_REAL[f"{k}_mean"].round(3)
    lsc[f"{k}|gen"] = LS_GEN[f"{k}_mean"].round(3)
    lsc[f"{k}|ratio"] = (LS_GEN[f"{k}_mean"] / LS_REAL[f"{k}_mean"].replace(0, np.nan)).round(3)
print("\n--- landscape metrics (per class, 128px, 8-connectivity) ---")
print(lsc.to_string(index=False))
print(f"Shannon diversity: real {SH_REAL.mean():.4f} +/- {SH_REAL.std():.4f} | "
      f"gen {SH_GEN.mean():.4f} +/- {SH_GEN.std():.4f}")

# ---- 3. thin-class connectivity ----
conn_rows, CONN_RAW = [], {}
for c, name in THIN.items():
    dr, dg = connectivity_stats(REAL128, c), connectivity_stats(GEN128, c)
    CONN_RAW[name] = (dr, dg)
    fr, fg = dr.frag.mean(), dg.frag.mean()
    sr, sg = dr.share.mean(), dg.share.mean()
    frag_ratio = fg / fr if fr else np.nan
    share_ratio = sg / sr if sr else np.nan
    # share_ratio now has an upper bound too: run A passed the old test while over-producing
    # River by 7.7x, because nothing checked for too MUCH of a thin class.
    ok = (frag_ratio == frag_ratio) and frag_ratio <= 3.0 and 0.25 <= share_ratio <= 2.5
    conn_rows.append(dict(cls=name, present_real=dr.present.mean(), present_gen=dg.present.mean(),
                          share_real=sr, share_gen=sg, ncomp_real=dr.n_comp.mean(),
                          ncomp_gen=dg.n_comp.mean(), frag_real=fr, frag_gen=fg,
                          meanarea_real=dr.mean_area.mean(), meanarea_gen=dg.mean_area.mean(),
                          largest_real=dr.largest_share.mean(), largest_gen=dg.largest_share.mean(),
                          frag_ratio=frag_ratio, share_ratio=share_ratio,
                          verdict="PASS" if ok else "FAIL"))
CONN = pd.DataFrame(conn_rows)
print("\n--- thin-class connectivity (the decisive check) ---")
print(CONN.round(4).to_string(index=False))

# ---- 4. memorization ----
nn_gen = nn_distance(GEN128[:cfg.NN_CHECK_N], TRAIN_128)
nn_val = nn_distance(VAL_128[:cfg.NN_CHECK_N], TRAIN_128)
mem_ratio = float(nn_gen.mean() / nn_val.mean())
print(f"\n--- memorization (min per-pixel disagreement vs the train set, 32x32) ---")
print(f"generated: mean {nn_gen.mean():.4f} min {nn_gen.min():.4f} | "
      f"held-out val baseline: mean {nn_val.mean():.4f} min {nn_val.min():.4f} | "
      f"ratio {mem_ratio:.3f}")
print("  ratio near or above 1.0 -> no memorization; well below -> the model is reciting tiles")

final = {"run": {"model": "MaskDDPM-UNet2D", "stage": 1, "arm": "B-onehot",
                 "label_coding": cfg.LABEL_CODING, "prediction_type": cfg.PREDICTION_TYPE,
                 "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
                 "train_sha1": split_sha1},
         "best_epoch": int(_ck["epoch"]), "select_on": cfg.SELECT_ON,
         "best_score": float(_ck.get("best_score", float("nan"))),
         "best_val_loss": float(_ck.get("val_loss", float("nan"))),
         "epochs_run": int(len(history)), "train_wall_clock_h_this_session": wall_h,
         "n_generated": int(len(GEN128)), "n_pool": int(len(RAW128)),
         "filtered": bool(cfg.FILTER_MASKS), "generation_minutes": gen_min,
         "postprocess": {"modal_k": PP_CFG[0], "close_k": PP_CFG[1],
                         "min_px_thin": PP_CFG[2], "min_px_bulk": PP_CFG[3]},
         "ambiguity_rate": float(ambiguity), "illegal_code_rate": 0.0,
         "KL_real_gen": KL_RG, "KL_real_gen_unfiltered": KL_RAW, "KL_gen_real": KL_GR, "JS": JS,
         "recon": RECON,
         "class_share_real": {CODE_NAMES[i]: float(p_real[i]) for i in range(cfg.N_CODES)},
         "class_share_gen": {CODE_NAMES[i]: float(p_gen[i]) for i in range(cfg.N_CODES)},
         "class_share_gen_unfiltered": {CODE_NAMES[i]: float(p_raw[i]) for i in range(cfg.N_CODES)},
         "shdi_real": float(SH_REAL.mean()), "shdi_gen": float(SH_GEN.mean()),
         "landscape": lsc.to_dict(orient="records"),
         "connectivity": CONN.to_dict(orient="records"),
         "memorization": {"nn_gen_mean": float(nn_gen.mean()), "nn_gen_min": float(nn_gen.min()),
                          "nn_val_mean": float(nn_val.mean()), "nn_val_min": float(nn_val.min()),
                          "ratio": mem_ratio},
         "peak_train_vram_gb": float(torch.cuda.max_memory_allocated() / 1e9)}
final.update(efficiency_report(best_model))
print("\nverdicts:", {r["cls"]: r["verdict"] for r in final["connectivity"]})


## 20. Persist results

Writes `results_{mode}_fold{k}.json` (full), a flat `.csv` row for the cross-stage comparison table,
and `history_{mode}_fold{k}.csv` for the curves — then pushes all three plus the model card to the
run's HF repo so they survive the session. `fold{FOLD}_split.json` goes up too: Stage 2 should pull it
from here rather than re-deriving it, so that any disagreement about the split becomes impossible
rather than merely unlikely.

In [ ]:
import csv
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2, default=str)

flat = {"model": "MaskDDPM-UNet2D", "arm": "B-onehot", "label_coding": cfg.LABEL_CODING,
        "prediction_type": cfg.PREDICTION_TYPE, "stage": 1, "split_mode": cfg.SPLIT_MODE,
        "fold": cfg.FOLD, "seed": cfg.SEED, "best_epoch": final["best_epoch"],
        "select_on": final["select_on"], "best_score": final["best_score"],
        "val_loss": final["best_val_loss"], "illegal_code_rate": final["illegal_code_rate"],
        "ambiguity_rate": final["ambiguity_rate"],
        "recon_pixel_acc": final.get("recon", {}).get("pixel_acc", float("nan")),
        "KL_real_gen": final["KL_real_gen"],
        "KL_real_gen_unfiltered": final["KL_real_gen_unfiltered"], "JS": final["JS"],
        "shdi_real": final["shdi_real"], "shdi_gen": final["shdi_gen"],
        "nn_ratio": final["memorization"]["ratio"],
        "params_M": final["params_M"], "GFLOPs_per_sample": final["GFLOPs_per_sample"],
        "masks_per_min": final["masks_per_min"], "peak_vram_gb": final["peak_train_vram_gb"],
        "n_generated": final["n_generated"], "train_sha1": split_sha1,
        "pp_modal": PP_CFG[0], "pp_close": PP_CFG[1], "pp_thin": PP_CFG[2],
        "pp_bulk": PP_CFG[3]}
for k, v in final["class_share_gen"].items(): flat[f"share_gen_{k}"] = v
for k, v in final["class_share_real"].items(): flat[f"share_real_{k}"] = v
for r in final["connectivity"]:
    flat[f"frag_ratio_{r['cls']}"] = r["frag_ratio"]
    flat[f"share_ratio_{r['cls']}"] = r["share_ratio"]
    flat[f"verdict_{r['cls']}"] = r["verdict"]

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
_hk = sorted({k for r in history for k in r})            # sel_* keys appear only on sampling epochs
with open(hp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_hk, restval=""); w.writeheader(); w.writerows(history)

lp = f"{cfg.OUT_DIR}/landscape_{tag}.csv"; lsc.to_csv(lp, index=False)
np_ = f"{cfg.OUT_DIR}/connectivity_{tag}.csv"; CONN.to_csv(np_, index=False)
pp = f"{cfg.OUT_DIR}/class_shares_{tag}.csv"; prop.to_csv(pp, index=False)

for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"), (hp, f"history_{tag}.csv"),
             (lp, f"landscape_{tag}.csv"), (np_, f"connectivity_{tag}.csv"),
             (pp, f"class_shares_{tag}.csv"),
             (f"{cfg.OUT_DIR}/{SPLIT_NAME}", SPLIT_NAME)]:
    hf_push(p, n, "results")
push_model_card(final, final["best_epoch"])

if WANDB:
    wandb.log({f"final/{k}": v for k, v in flat.items() if isinstance(v, (int, float))})
    wandb.finish()
print("results pushed to", HF_REPO)
print("\nFor the ablation table, put this row next to run A's and compare on: KL_real_gen, "
      "share_gen_River, frag_ratio_River, verdict_River, illegal/ambiguity, downstream mIoU.")


## 21. Stage-2 hand-off package

**This is the interface contract. Do not change anything in this cell without telling Stage 2.**

Each generated mask is upsampled 128 -> 512 with `INTER_NEAREST` (exact for label maps; any other
kernel would invent classes along every boundary), code 6 is mapped back to **255**, and the result
is written as a single-channel PNG named `genmask{i}.png`.

The asserts below re-read a written file and check it against the contract: 2-D array, `uint8`,
512x512, values in `{0..5, 255}`. That is deliberately paranoid — a silent 3-channel write here
would not surface until Stage 2's dataloader produced garbage a week later.

The whole set is tarred and pushed alongside a manifest carrying the split SHA1 and the architecture
signature, so the masks can always be traced back to the exact model and split that produced them.

In [ ]:
for f in Path(cfg.GEN_DIR).glob("genmask*.png"): f.unlink()          # never mix two runs
t0 = time.time()
for i, m in enumerate(GEN128):
    out = cv2.resize(m, (cfg.OUT_SIZE, cfg.OUT_SIZE), interpolation=cv2.INTER_NEAREST)
    out = np.where(out == cfg.IGNORE_CODE, cfg.IGNORE_INDEX, out).astype(np.uint8)  # 6 -> 255
    cv2.imwrite(f"{cfg.GEN_DIR}/genmask{i}.png", out, [cv2.IMWRITE_PNG_COMPRESSION, 3])
written = sorted(Path(cfg.GEN_DIR).glob("genmask*.png"))
print(f"wrote {len(written)} masks to {cfg.GEN_DIR} in {time.time()-t0:.0f}s")

# ---- contract check ----
chk = cv2.imread(f"{cfg.GEN_DIR}/genmask0.png", cv2.IMREAD_UNCHANGED)
assert chk.ndim == 2, f"genmask0.png is not single channel: {chk.shape}"
assert chk.dtype == np.uint8, chk.dtype
assert chk.shape == (cfg.OUT_SIZE, cfg.OUT_SIZE), chk.shape
assert len(written) == cfg.N_GENERATE, f"{len(written)} files, expected {cfg.N_GENERATE}"
vals = set()
for p in written[::37]:
    vals |= set(np.unique(cv2.imread(str(p), cv2.IMREAD_UNCHANGED)).tolist())
assert vals <= set(range(cfg.N_CLASSES)) | {cfg.IGNORE_INDEX}, f"illegal values on disk: {sorted(vals)}"
print(f"contract OK | shape {chk.shape} dtype {chk.dtype} | values seen {sorted(vals)}")

manifest = {"produced_by": RUN_NAME, "hf_repo": HF_REPO, "arch": ARCH_SIG,
            "best_epoch": final["best_epoch"], "n_masks": len(written),
            "size": cfg.OUT_SIZE, "upsample": "INTER_NEAREST from 128",
            "values": "0..5 semantic, 255 ignore", "naming": "genmask{i}.png",
            "class_names": list(cfg.CLASS_NAMES), "train_sha1": split_sha1,
            "class_share_gen": final["class_share_gen"],
            "connectivity_verdicts": {r["cls"]: r["verdict"] for r in final["connectivity"]}}
mp = f"{cfg.OUT_DIR}/genmask_manifest.json"
json.dump(manifest, open(mp, "w"), indent=2)

TAR = f"{cfg.OUT_DIR}/gen_masks_{cfg.OUT_SIZE}_{tag}.tar.gz"
with tarfile.open(TAR, "w:gz") as tf:
    tf.add(cfg.GEN_DIR, arcname=f"gen_masks_{cfg.OUT_SIZE}")
hf_push(mp, "genmask_manifest.json", "hand-off manifest")
hf_push(TAR, os.path.basename(TAR), "generated masks for Stage 2")
print(f"archive: {TAR} ({os.path.getsize(TAR)/1e6:.1f} MB) -> pushed to {HF_REPO}")
print("\nStage 2 input: pull the archive above, or Save Version and attach "
      f"{cfg.GEN_DIR} as a Kaggle dataset.")

## 22. Standalone sampling from Hugging Face

Self-contained, exactly like the inference cell in the segmentation notebooks: point `GEN_REPO` at
this run's repo, ask for N masks, get 512x512 label PNGs back. Nothing above this cell is required —
the UNet is rebuilt from the **full config embedded in `best.pt`**, so the architecture is
reconstructed exactly as trained, and the binary codes are re-derived locally from `N_CODES`.

Paste it into a fresh notebook as-is if Stage 2 ever needs more masks than the hand-off set contains,
or a second set drawn from a different seed for a variance estimate.

In [ ]:
def load_generator_from_hf(repo_id, token=None, device="cuda"):
    import torch
    from diffusers import UNet2DModel
    from huggingface_hub import hf_hub_download
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location=device)
    m = UNet2DModel.from_config(ck["model_config"]).to(device).eval()
    m.load_state_dict(ck["model_state"])
    return m, ck["cfg"], ck

@torch.no_grad()
def generate_masks_from_hf(m, c, n, steps=50, seed=0, batch=25, device="cuda"):
    import numpy as np, cv2, torch
    from diffusers import DDIMScheduler
    assert c.get("LABEL_CODING", "onehot") == "onehot", \
        "this sampler decodes one-hot; a binary-code checkpoint needs run A's sampler"
    nch = c["N_CH"]
    out = []
    for s in range(0, n, batch):
        b = min(batch, n - s)
        gg = torch.Generator(); gg.manual_seed(seed + s)
        x = torch.randn(b, nch, c["GEN_SIZE"], c["GEN_SIZE"], generator=gg).to(device)
        kw = dict(num_train_timesteps=c["TIMESTEPS"], beta_start=c["BETA_START"],
                  beta_end=c["BETA_END"], beta_schedule="linear",
                  prediction_type=c["PREDICTION_TYPE"], clip_sample=True, steps_offset=0)
        if c.get("ZERO_TERMINAL_SNR", False):
            kw.update(rescale_betas_zero_snr=True, timestep_spacing="trailing",
                      set_alpha_to_one=True)
        else:
            kw.update(set_alpha_to_one=False)
        sc = DDIMScheduler(**kw)
        sc.set_timesteps(steps)
        for t in sc.timesteps:
            with torch.cuda.amp.autocast():
                pred = m(x, t).sample
            x = sc.step(pred.float(), t, x, eta=0.0).prev_sample
        idx = x.clamp(-1, 1).argmax(dim=1)                  # one-hot: argmax, no illegal state
        a = idx.to(torch.uint8).cpu().numpy()
        big = np.stack([cv2.resize(z, (c["OUT_SIZE"], c["OUT_SIZE"]),
                                   interpolation=cv2.INTER_NEAREST) for z in a])
        out.append(np.where(big == c["IGNORE_CODE"], c["IGNORE_INDEX"], big).astype(np.uint8))
    return np.concatenate(out)

# --- example ---
GEN_REPO = HF_REPO
gen_model, gen_cfg, gen_ck = load_generator_from_hf(GEN_REPO, token=HF_TOKEN)
demo = generate_masks_from_hf(gen_model, gen_cfg, 4, steps=cfg.DDIM_STEPS, seed=99)
print(f"loaded {GEN_REPO} (epoch {gen_ck['epoch']}) | demo {demo.shape} "
      f"values {sorted(np.unique(demo).tolist())}")
_f = mask_grid(demo, ncol=4, title="standalone sampler - 512 output"); plt.show()


## 23. Qualitative grids — generated vs real

The counterpart of the segmentation notebooks' triptychs. Same palette, ignore renders black, so
these figures sit next to those in the paper without a second legend.

What to look for, in order:
1. **Do the layouts read as Kathmandu?** Residential matrix with agricultural patches on the fringe,
   forest on the ridges, a river thread crossing the tile. Plausible topology matters more than
   plausible texture — texture is Stage 2's job.
2. **Are roads continuous lines or dotted noise?** This is the eyeball version of the `frag` number;
   the two should agree, and if they disagree trust the number.
3. **Are the class boundaries crisp?** Binary coding plus nearest-code decoding should give hard
   edges. Soft, speckled boundaries mean the samples are still far from the code corners — check the
   illegal-code rate and, if it is high, train longer before blaming the representation.

The second figure isolates Road and River as binary masks, because at 128 px on a 7-colour palette a
thin river is genuinely hard to see against Forest.

In [ ]:
fig = mask_grid(GEN128[:16], ncol=4, title=f"GENERATED - {RUN_NAME}, epoch {final['best_epoch']}")
pg = f"{cfg.OUT_DIR}/qual_generated_{tag}.png"
plt.savefig(pg, dpi=130, bbox_inches="tight"); plt.show()

fig = mask_grid(REAL128[:16], ncol=4, title=f"REAL - fold-{cfg.FOLD} train masks")
pr = f"{cfg.OUT_DIR}/qual_real_{tag}.png"
plt.savefig(pr, dpi=130, bbox_inches="tight"); plt.show()

# ---- thin classes isolated ----
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for j in range(8):
    axes[0, j].imshow(np.isin(REAL128[j], [1, 2]) * (REAL128[j] == 1) * 1.0 +
                      np.isin(REAL128[j], [1, 2]) * (REAL128[j] == 2) * 0.5, cmap="gray")
    axes[0, j].set_title(f"real {j}", fontsize=8); axes[0, j].axis("off")
    axes[1, j].imshow((GEN128[j] == 1) * 1.0 + (GEN128[j] == 2) * 0.5, cmap="gray")
    axes[1, j].set_title(f"gen {j}", fontsize=8); axes[1, j].axis("off")
fig.suptitle("Road (white) and River (grey) only - real top, generated bottom", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.95])
pt = f"{cfg.OUT_DIR}/qual_thin_{tag}.png"
plt.savefig(pt, dpi=130, bbox_inches="tight"); plt.show()

for p in [pg, pr, pt]: hf_push(p, os.path.basename(p), "qualitative")

## 24. Distribution, landscape and connectivity figures

Three panels, in the order a reader should meet them:

1. **Class proportions**, real vs generated, on a log axis — without the log axis River at 0.57%
   is invisible next to Residential at 42%, which is precisely the class the figure exists to show.
2. **Landscape metric ratios** (generated / real) per class. A bar at 1.0 is a perfect match; bars
   far above 1 mean over-fragmentation, far below mean over-smoothing. This is the single most
   informative panel for judging whether the samples are geographically plausible rather than merely
   correctly coloured.
3. **Connectivity histograms** for Road and River — the distribution of components per 1000 class
   pixels, real against generated. Overlapping distributions are the pass condition; a generated
   distribution shifted right is the dotted-river failure, stated as a distribution rather than a
   single mean.

In [ ]:
x = np.arange(cfg.N_CODES); w = 0.28
fig, ax = plt.subplots(figsize=(9.0, 3.4))
ax.bar(x - w, 100 * p_real, w, label="real")
ax.bar(x,     100 * p_raw,  w, label="generated (unfiltered)")
ax.bar(x + w, 100 * p_gen,  w, label="generated (hand-off set)")
ax.set_xticks(x); ax.set_xticklabels(CODE_NAMES, rotation=20, ha="right")
ax.set_ylabel("pixel share (%)"); ax.set_yscale("log")
ax.set_title(f"Class proportions - KL(real||gen) = {KL_RG:.4f} filtered / {KL_RAW:.4f} raw, "
             f"JS = {JS:.4f}")
ax.legend(fontsize=8); plt.tight_layout()
p1 = f"{cfg.OUT_DIR}/fig_class_proportions_{tag}.png"
plt.savefig(p1, dpi=140, bbox_inches="tight"); plt.show()

keys = ["n_patches", "mean_area", "edge_density", "largest_share"]
fig, ax = plt.subplots(figsize=(9.5, 3.4))
bw = 0.2
for i, k in enumerate(keys):
    r = (LS_GEN[f"{k}_mean"] / LS_REAL[f"{k}_mean"].replace(0, np.nan)).values
    ax.bar(np.arange(cfg.N_CODES) + (i - 1.5) * bw, r, bw, label=k)
ax.axhline(1.0, color="k", lw=1, ls="--")
ax.set_xticks(np.arange(cfg.N_CODES)); ax.set_xticklabels(CODE_NAMES, rotation=20, ha="right")
ax.set_ylabel("generated / real"); ax.set_yscale("log")
ax.set_title(f"Landscape metric ratios (1.0 = perfect match) | SHDI real {SH_REAL.mean():.3f} "
             f"vs gen {SH_GEN.mean():.3f}")
ax.legend(fontsize=8, ncol=4); plt.tight_layout()
p2 = f"{cfg.OUT_DIR}/fig_landscape_{tag}.png"
plt.savefig(p2, dpi=140, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (c, name) in zip(axes[:2], THIN.items()):
    dr, dg = CONN_RAW[name]
    ax.hist(dr.frag.dropna(), bins=30, alpha=0.55, density=True, label="real")
    ax.hist(dg.frag.dropna(), bins=30, alpha=0.55, density=True, label="generated")
    v = CONN[CONN.cls == name].iloc[0]
    ax.set_title(f"{name}: components / 1000 px\nfrag {v.frag_ratio:.2f}, share "
                 f"{v.share_ratio:.2f} -> {v.verdict}", fontsize=9)
    ax.set_xlabel("frag"); ax.legend(fontsize=8)
axes[2].hist(nn_val, bins=30, alpha=0.55, density=True, label="held-out val vs train")
axes[2].hist(nn_gen, bins=30, alpha=0.55, density=True, label="generated vs train")
axes[2].set_title(f"NN distance to train set\nratio {mem_ratio:.2f}", fontsize=9)
axes[2].set_xlabel("min per-pixel disagreement"); axes[2].legend(fontsize=8)
plt.tight_layout()
p3 = f"{cfg.OUT_DIR}/fig_connectivity_{tag}.png"
plt.savefig(p3, dpi=140, bbox_inches="tight"); plt.show()

# ---- selection-metric curves: the series that run A did not have ----
_hs = [r for r in history if "sel_kl" in r]
if _hs:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    e = [r["epoch"] for r in _hs]
    axes[0].plot(e, [r["sel_kl"] for r in _hs]); axes[0].set_yscale("log")
    axes[0].set_title("sampled class-share KL (the selector)", fontsize=9)
    axes[1].plot(e, [100 * r["sel_share_River"] for r in _hs], label="River")
    axes[1].plot(e, [100 * r["sel_share_Road"] for r in _hs], label="Road")
    axes[1].axhline(100 * P_REAL_TRAIN[2], ls="--", lw=1, color="C0")
    axes[1].axhline(100 * P_REAL_TRAIN[1], ls="--", lw=1, color="C1")
    axes[1].set_title("thin-class share vs real (dashed)", fontsize=9); axes[1].legend(fontsize=8)
    axes[2].plot([r["epoch"] for r in history], [r["val_loss"] for r in history], label="val")
    axes[2].plot([r["epoch"] for r in history], [r["val_loss_lowt"] for r in history],
                 label=f"val t<{cfg.VAL_LOWT}")
    axes[2].set_yscale("log"); axes[2].set_title("denoising loss (NOT the selector)", fontsize=9)
    axes[2].legend(fontsize=8)
    for a in axes: a.set_xlabel("epoch")
    plt.tight_layout()
    p4 = f"{cfg.OUT_DIR}/fig_selection_{tag}.png"
    plt.savefig(p4, dpi=140, bbox_inches="tight"); plt.show()
    hf_push(p4, os.path.basename(p4), "figures")

for p in [p1, p2, p3]: hf_push(p, os.path.basename(p), "figures")
print("\n--- STAGE 1 SUMMARY ---")
print(json.dumps({k: final[k] for k in ["best_epoch", "select_on", "best_score", "best_val_loss",
                                        "n_generated", "ambiguity_rate", "KL_real_gen",
                                        "KL_real_gen_unfiltered", "JS", "shdi_real", "shdi_gen",
                                        "masks_per_min"]}, indent=2, default=str))
print("recon:", {k: (round(v, 4) if isinstance(v, float) else v)
                 for k, v in final.get("recon", {}).items() if k != "per_class_iou"})
print("connectivity:", {r["cls"]: f"{r['verdict']} (frag x{r['frag_ratio']:.2f}, "
                                 f"share x{r['share_ratio']:.2f})"
                        for r in final["connectivity"]})
print(f"memorization ratio: {mem_ratio:.3f}")
print(f"hand-off: {cfg.GEN_DIR}/genmask*.png -> Stage 2")


## 25. Run checklist

1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.
2. `cfg.HF_PREFIX` ends in `-onehot`, so this arm gets its own repo and cannot overwrite run A.
3. Section 5 must print **channel-order check PASSED** and an unmapped share of `0.000000%`.
4. Section 6 repacks to the index cache (~10-15 min for 1224 masks at 2345x2351). Then
   `Save Version` and attach the cache as a Kaggle dataset, and set `PREBUILT_CACHE` next session.
5. Section 9 must print an exact one-hot round trip, and the degradation table showing the binary
   code losing pixels to non-boundary classes and to code 7 at the same noise level. That table is
   ablation-A-vs-B evidence — keep the printed output.
6. Section 11 asserts v-prediction against zero terminal SNR, and round-trips v -> x0 at five
   timesteps. If either trips, stop; do not train.
7. Section 10 asserts the one-hot invariant (exactly one `+1` per pixel) on a real batch.
8. **Watch `sel_kl` and `sel_share_River` in W&B, not `val_loss`.** Section 10 now prints the real
   `frac_with_River` / `frac_with_Road` / SHDI baselines, so compare the `sel_*` series against
   those, not against nothing.
9. **Resuming a run that is already past epoch 1100:** `MAX_EPOCHS` and `COS_HORIZON_EPOCHS` are
   both 1600. The cosine is rebuilt from the new horizon on resume, so the learning rate drops
   sharply at the first step of the new session (roughly 1.2e-4 to 5e-5). That is the annealing
   this run was missing, not a bug. About 500 epochs, ~4.5 h, one session.
10. `RESET_BEST_ON_RESUME=True` re-opens the best-checkpoint contest once, because `SEL_N_SAMPLE`
    changed from 64 to 128 and the two are not the same estimator. `best.pt` already on the hub is
    left untouched, so a session that never beats it still leaves you a usable checkpoint. Set this
    back to `False` if you ever resume again without changing `SEL_N_SAMPLE`.
11. The River gate means some sampling epochs will print `not eligible` instead of setting a new
    best. That is the gate working. If **every** epoch past e1200 is ineligible, River has genuinely
    collapsed — stop and read the fallback list below.
12. Read the reconstruction accuracy in section 19 before drawing any conclusion from the grid:
    - high accuracy, bad samples -> prior/schedule, not capacity,
    - low accuracy -> still undertrained, extend `MAX_EPOCHS` and `COS_HORIZON_EPOCHS` together.
13. Section 14 now prints how long the section-19 pool will take at `FINAL_DDIM_STEPS`. Check it
    before you commit the session: `N_POOL=3500` at 200 steps is not free.
14. Section 21 writes the Stage-2 hand-off set. The contract is unchanged from run A, so Stage 2
    needs no edits: single-channel 512x512 PNGs, values `{0..5, 255}`, `genmask{i}.png`.
15. For the paper, put `results_block_fold1.csv` from both arms side by side. The columns that
    carry the argument: `KL_real_gen_unfiltered`, `share_gen_River`, `frag_ratio_River`,
    `verdict_River`, `illegal_code_rate` vs `ambiguity_rate`, and later the stage-4 mIoU delta.

### If River still comes out wrong after this
In order, and one at a time:
1. Read `recon_pixel_acc`. Under ~0.90 means undertrained — more steps before anything else.
2. Raise `FINAL_DDIM_STEPS` to 500 and re-sample from the same checkpoint. If River improves, it
   was a sampling-step problem, which is cheap to fix and worth a sentence in the paper.
3. Only then raise `GEN_SIZE` to 192 or 256 (and `ATTN_RES` to match). At 128 a River is ~3 px
   wide; there may simply be no room. Budget 4x the wall clock at 256.
4. `RIVER_OVERSAMPLE` last, and only if River is *under*-produced. It biases the learned marginal
   and must be declared.
